In [1]:
# -*- coding: utf-8 -*-
import pandas as pd
from sklearn.model_selection import train_test_split

# ===================== 配置 =====================
input_file = 'ruc_Class25Q2_train_rent.csv'
output_file = 'ruc_Class25Q2_train_rent1.csv'

test_size = 0.2
random_state = 42

# 需要保留的列
keep_columns = [
    '城市', '户型', 'Price', '楼层', '面积', '朝向', 
    '交易时间', '付款方式', '租赁方式', '电梯', '租期', '配套设施', 
    'lon', 'lat', '板块', '建筑年代', '房屋总数', '楼栋总数', 
    '绿 化 率', '容 积 率', '物 业 费'
]

# =================================================

print("=" * 80)
print("租金数据预处理")
print("=" * 80)

# 1. 读取数据
df = pd.read_csv(input_file, encoding='utf-8-sig')
print(f"\n原始数据: {df.shape}")

# 2. 筛选列（自动处理空格）
existing_columns = []
for col in keep_columns:
    if col in df.columns:
        existing_columns.append(col)
    else:
        # 匹配去空格版本
        col_no_space = col.replace(' ', '')
        candidates = [c for c in df.columns if c.replace(' ', '') == col_no_space]
        if candidates:
            existing_columns.append(candidates[0])

df_filtered = df[existing_columns].copy()

# 去除列名空格
df_filtered.columns = df_filtered.columns.str.replace(' ', '')

print(f"保留列数: {len(df_filtered.columns)}")

# 3. 分割训练/测试集（分层抽样）
if 'Price' in df_filtered.columns and df_filtered['Price'].notna().sum() > 0:
    df_filtered['_price_bin'] = pd.qcut(
        df_filtered['Price'].fillna(df_filtered['Price'].median()), 
        q=5, labels=False, duplicates='drop'
    )
    train_idx, test_idx = train_test_split(
        df_filtered.index, test_size=test_size, 
        random_state=random_state, stratify=df_filtered['_price_bin']
    )
    df_filtered.drop('_price_bin', axis=1, inplace=True)
    print("✅ 分层抽样完成")
else:
    train_idx, test_idx = train_test_split(
        df_filtered.index, test_size=test_size, random_state=random_state
    )
    print("✅ 随机抽样完成")

# 4. 添加is_train标识
df_filtered['is_train'] = 0
df_filtered.loc[train_idx, 'is_train'] = 1

print(f"训练集: {(df_filtered['is_train']==1).sum():,} ({(df_filtered['is_train']==1).sum()/len(df_filtered)*100:.1f}%)")
print(f"测试集: {(df_filtered['is_train']==0).sum():,} ({(df_filtered['is_train']==0).sum()/len(df_filtered)*100:.1f}%)")

# 5. 保存
df_filtered.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"\n✅ 已保存: {output_file}")
print(f"最终数据: {df_filtered.shape}")
print("=" * 80)

租金数据预处理


C:\Users\Administrator\AppData\Local\Temp\ipykernel_21440\1224104201.py:27: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file, encoding='utf-8-sig')



原始数据: (98899, 46)
保留列数: 21
✅ 分层抽样完成
训练集: 79,119 (80.0%)
测试集: 19,780 (20.0%)

✅ 已保存: ruc_Class25Q2_train_rent1.csv
最终数据: (98899, 22)


In [2]:
import pandas as pd

# 读取数据
df = pd.read_csv('ruc_Class25Q2_train_rent1.csv')

print(f"删除前: {df.shape}")

# 删除租期列
df_clean = df.drop(columns=['租期'])

print(f"删除后: {df_clean.shape}")
print(f"剩余列: {list(df_clean.columns)}")

# 保存新文件
df_clean.to_csv('ruc_Class25Q2_train_rent2.csv', index=False, encoding='utf-8-sig')
print("\n✅ 已生成: ruc_Class25Q2_train_rent2.csv")

删除前: (98899, 22)
删除后: (98899, 21)
剩余列: ['城市', '户型', 'Price', '楼层', '面积', '朝向', '交易时间', '付款方式', '租赁方式', '电梯', '配套设施', 'lon', 'lat', '板块', '建筑年代', '房屋总数', '楼栋总数', '绿化率', '容积率', '物业费', 'is_train']

✅ 已生成: ruc_Class25Q2_train_rent2.csv


In [3]:
# -*- coding: utf-8 -*-
"""
租房数据清洗 - 户型和面积处理（训练/测试集同文件）
"""

import pandas as pd
import numpy as np
import re

# ===================== 配置 =====================
path_in = r"H:\HW\ruc_Class25Q2_train_rent2.csv"
path_out = r"H:\HW\ruc_Class25Q2_train_rent3.csv"

# =================================================

print("=" * 80)
print("租房数据清洗 - 户型和面积")
print("=" * 80)

# ============ 1. 加载 ============
df = pd.read_csv(path_in, encoding='utf-8-sig')
print(f"\n原始数据: {df.shape}")

if 'is_train' in df.columns:
    print(f"训练集: {(df['is_train']==1).sum():,}, 测试集: {(df['is_train']==0).sum():,}")

# 找列
layout_col = [c for c in df.columns if '户型' in c][0]
area_col = [c for c in df.columns if '面积' in c][0]

print(f"户型列: {layout_col}, 面积列: {area_col}")


# ============ 2. 清洗面积 ============
print("\n" + "-" * 80)
print("清洗面积")

def clean_area(s):
    if pd.isna(s):
        return np.nan
    s = str(s).replace('m²', '').replace('㎡', '').replace('平方米', '').replace('平米', '').strip()
    m = re.search(r'(\d+\.?\d*)', s)
    return float(m.group(1)) if m else np.nan

df['建筑面积'] = df[area_col].apply(clean_area)
print(f"✅ 有效: {df['建筑面积'].notna().sum()}/{len(df)}")


# ============ 3. 解析户型 ============
print("\n" + "-" * 80)
print("解析户型")

def parse_layout(s):
    if pd.isna(s):
        return np.nan, np.nan, np.nan
    s = str(s)
    
    shi = ting = wei = np.nan
    m = re.search(r'(\d+)[室房]', s)
    if m: shi = int(m.group(1))
    m = re.search(r'(\d+)厅', s)
    if m: ting = int(m.group(1))
    m = re.search(r'(\d+)卫', s)
    if m: wei = int(m.group(1))
    
    return shi, ting, wei

df[['室_tmp', '厅_tmp', '卫_tmp']] = df[layout_col].apply(lambda x: pd.Series(parse_layout(x)))

print(f"✅ 室: {df['室_tmp'].notna().sum()}, 厅: {df['厅_tmp'].notna().sum()}, 卫: {df['卫_tmp'].notna().sum()}")


# ============ 4. 创建最终变量 ============
print("\n" + "-" * 80)
print("创建最终变量")

# 室 = 室 + 厅
df['室'] = df['室_tmp'].fillna(0) + df['厅_tmp'].fillna(0)
df['室'] = df['室'].replace(0, np.nan)

# 卫
df['卫'] = df['卫_tmp']

# 面积
df['建筑面积_final'] = df['建筑面积']

print(f"室: {df['室'].notna().sum()}, 卫: {df['卫'].notna().sum()}, 面积: {df['建筑面积_final'].notna().sum()}")


# ============ 5. 🔥 仅对训练集计算中位数，填充训练集和测试集 ============
print("\n" + "-" * 80)
print("中位数填充（基于训练集）")

if 'is_train' in df.columns:
    # 🔥 只用训练集计算中位数
    train_mask = df['is_train'] == 1
    
    # 室
    shi_median = df.loc[train_mask, '室'].median()
    df['室'].fillna(shi_median, inplace=True)
    print(f"室: 中位数={shi_median:.0f}")
    
    # 卫
    wei_median = df.loc[train_mask, '卫'].median()
    df['卫'].fillna(wei_median, inplace=True)
    print(f"卫: 中位数={wei_median:.0f}")
    
    # 面积
    area_median = df.loc[train_mask, '建筑面积_final'].median()
    df['建筑面积_final'].fillna(area_median, inplace=True)
    print(f"面积: 中位数={area_median:.1f}")
    
    print("\n✅ 已用训练集中位数填充所有缺失值")
else:
    # 如果没有is_train，用全局中位数
    df['室'].fillna(df['室'].median(), inplace=True)
    df['卫'].fillna(df['卫'].median(), inplace=True)
    df['建筑面积_final'].fillna(df['建筑面积_final'].median(), inplace=True)
    print("✅ 已用全局中位数填充")


# ============ 6. 删除临时列 ============
df.drop(['室_tmp', '厅_tmp', '卫_tmp', '建筑面积'], axis=1, inplace=True)

# 重命名
df.rename(columns={'建筑面积_final': '建筑面积'}, inplace=True)


# ============ 7. 异常值处理 ============
print("\n" + "-" * 80)
print("异常值检查")

n_before = len(df)

# 室 > 10
abnormal_shi = df['室'] > 10
if abnormal_shi.sum() > 0:
    print(f"⚠️ 异常室数(>10): {abnormal_shi.sum()}")
    df = df[~abnormal_shi]

# 面积 < 10 或 > 500
abnormal_area = (df['建筑面积'] < 10) | (df['建筑面积'] > 500)
if abnormal_area.sum() > 0:
    print(f"⚠️ 异常面积(<10或>500): {abnormal_area.sum()}")
    df = df[~abnormal_area]

print(f"删除: {n_before - len(df)}")


# ============ 8. 保存 ============
print("\n" + "=" * 80)
print("保存")
print("=" * 80)

# 列顺序：建筑面积、室、卫在前
cols = df.columns.tolist()
new_order = []

for c in ['建筑面积', '室', '卫']:
    if c in cols:
        new_order.append(c)

for c in cols:
    if c not in new_order:
        new_order.append(c)

df = df[new_order]

df.to_csv(path_out, index=False, encoding='utf-8-sig')

print(f"最终数据: {df.shape}")
print(f"✅ 已保存: {path_out}")

if 'is_train' in df.columns:
    print(f"\n训练集: {(df['is_train']==1).sum():,}")
    print(f"测试集: {(df['is_train']==0).sum():,}")

print("\n" + "=" * 80)
print("✨ 完成")
print("=" * 80)

租房数据清洗 - 户型和面积

原始数据: (98899, 21)
训练集: 79,119, 测试集: 19,780
户型列: 户型, 面积列: 面积

--------------------------------------------------------------------------------
清洗面积
✅ 有效: 98899/98899

--------------------------------------------------------------------------------
解析户型
✅ 室: 95116, 厅: 93871, 卫: 34765

--------------------------------------------------------------------------------
创建最终变量
室: 95122, 卫: 34765, 面积: 98899

--------------------------------------------------------------------------------
中位数填充（基于训练集）
室: 中位数=4
卫: 中位数=1
面积: 中位数=79.0

✅ 已用训练集中位数填充所有缺失值

--------------------------------------------------------------------------------
异常值检查
⚠️ 异常室数(>10): 17
⚠️ 异常面积(<10或>500): 1173
删除: 1190

保存


C:\Users\Administrator\AppData\Local\Temp\ipykernel_21440\2949636346.py:100: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['室'].fillna(shi_median, inplace=True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21440\2949636346.py:105: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a co

最终数据: (97709, 24)
✅ 已保存: H:\HW\ruc_Class25Q2_train_rent3.csv

训练集: 78,184
测试集: 19,525

✨ 完成


In [4]:
# -*- coding: utf-8 -*-
"""
租房数据清洗 - 简单调整
"""

import pandas as pd

# ===================== 配置 =====================
path_in = r"H:\HW\ruc_Class25Q2_train_rent3.csv"
path_out = r"H:\HW\ruc_Class25Q2_train_rent4.csv"

# 重命名配置
rename_dict = {
    '建筑面积': '面积'
}

# 删除列配置
drop_cols = ['户型','卫']

# =================================================

print("=" * 80)
print("租房数据清洗 - 简单调整")
print("=" * 80)

# 加载
df = pd.read_csv(path_in, encoding='utf-8-sig')
print(f"\n原始数据: {df.shape}")
print(f"列名: {df.columns.tolist()}")

# 重命名
if rename_dict:
    existing_rename = {k: v for k, v in rename_dict.items() if k in df.columns}
    if existing_rename:
        df.rename(columns=existing_rename, inplace=True)
        print(f"\n✅ 已重命名: {existing_rename}")
    else:
        print(f"\n⚠️ 未找到需要重命名的列: {list(rename_dict.keys())}")

# 删除列
existing_drop_cols = [c for c in drop_cols if c in df.columns]
if existing_drop_cols:
    df.drop(columns=existing_drop_cols, inplace=True)
    print(f"✅ 已删除列: {existing_drop_cols}")
else:
    print(f"⚠️ 未找到需要删除的列: {drop_cols}")

# 保存
df.to_csv(path_out, index=False, encoding='utf-8-sig')

print(f"\n最终数据: {df.shape}")
print(f"最终列名: {df.columns.tolist()}")
print(f"✅ 已保存: {path_out}")

if 'is_train' in df.columns:
    print(f"\n训练集: {(df['is_train']==1).sum():,}")
    print(f"测试集: {(df['is_train']==0).sum():,}")

print("\n" + "=" * 80)
print("✨ 完成")
print("=" * 80)

租房数据清洗 - 简单调整

原始数据: (97709, 24)
列名: ['建筑面积', '室', '卫', '城市', '户型', 'Price', '楼层', '面积', '朝向', '交易时间', '付款方式', '租赁方式', '电梯', '配套设施', 'lon', 'lat', '板块', '建筑年代', '房屋总数', '楼栋总数', '绿化率', '容积率', '物业费', 'is_train']

✅ 已重命名: {'建筑面积': '面积'}
✅ 已删除列: ['户型', '卫']

最终数据: (97709, 22)
最终列名: ['面积', '室', '城市', 'Price', '楼层', '面积', '朝向', '交易时间', '付款方式', '租赁方式', '电梯', '配套设施', 'lon', 'lat', '板块', '建筑年代', '房屋总数', '楼栋总数', '绿化率', '容积率', '物业费', 'is_train']
✅ 已保存: H:\HW\ruc_Class25Q2_train_rent4.csv

训练集: 78,184
测试集: 19,525

✨ 完成


In [5]:
# -*- coding: utf-8 -*-
"""
租房数据 - 处理朝向和交易时间
"""

import pandas as pd
import numpy as np

# ===================== 配置 =====================
path = r"H:\HW\ruc_Class25Q2_train_rent4.csv"
out_path = r"H:\HW\ruc_Class25Q2_train_rent5.csv"
# =================================================

print("处理朝向和交易时间")
print("=" * 80)

# 读取
df = pd.read_csv(path, encoding='utf-8-sig')
print(f"原始数据: {df.shape}")

if 'is_train' in df.columns:
    train_mask = df['is_train'] == 1
    print(f"训练集: {train_mask.sum():,}, 测试集: {(~train_mask).sum():,}")
else:
    train_mask = pd.Series([True] * len(df))
    print("⚠️ 未找到is_train列，使用全部数据")

# 找朝向列和交易时间列
direction_col = [col for col in df.columns if '朝向' in col or 'direction' in col.lower()]
time_col = [col for col in df.columns if '交易时间' in col or '时间' in col or 'time' in col.lower() or 'date' in col.lower()]

direction_col = direction_col[0] if direction_col else None
time_col = time_col[0] if time_col else None

print(f"朝向列: '{direction_col}'")
print(f"交易时间列: '{time_col}'")


# ============ 1. 处理朝向 ============
if direction_col:
    print("\n" + "-" * 80)
    print("处理朝向（东南西北01变量）")
    print("-" * 80)
    
    print(f"\n原始朝向样例（前20个）:")
    print(df[direction_col].value_counts().head(20))
    
    # 创建四个方向的01变量
    df['朝向_东'] = df[direction_col].astype(str).str.contains('东', na=False).astype(int)
    df['朝向_南'] = df[direction_col].astype(str).str.contains('南', na=False).astype(int)
    df['朝向_西'] = df[direction_col].astype(str).str.contains('西', na=False).astype(int)
    df['朝向_北'] = df[direction_col].astype(str).str.contains('北', na=False).astype(int)
    
    print(f"\n朝向01变量统计:")
    print(f"  东: {df['朝向_东'].sum()} ({df['朝向_东'].sum()/len(df)*100:.1f}%)")
    print(f"  南: {df['朝向_南'].sum()} ({df['朝向_南'].sum()/len(df)*100:.1f}%)")
    print(f"  西: {df['朝向_西'].sum()} ({df['朝向_西'].sum()/len(df)*100:.1f}%)")
    print(f"  北: {df['朝向_北'].sum()} ({df['朝向_北'].sum()/len(df)*100:.1f}%)")
    
    print(f"\n样例（前10个）:")
    print(df[[direction_col, '朝向_东', '朝向_南', '朝向_西', '朝向_北']].head(10).to_string(index=False))


# ============ 2. 处理交易时间 ============
if time_col:
    print("\n" + "-" * 80)
    print("处理交易时间（年月one-hot编码）")
    print("-" * 80)
    
    print(f"\n原始交易时间样例:")
    print(df[time_col].head(10).tolist())
    
    # 转换为datetime
    df['交易时间_dt'] = pd.to_datetime(df[time_col], errors='coerce')
    
    # 提取年和月
    df['年'] = df['交易时间_dt'].dt.year
    df['月'] = df['交易时间_dt'].dt.month
    
    print(f"\n年份分布:")
    print(df['年'].value_counts().sort_index())
    
    print(f"\n月份分布:")
    print(df['月'].value_counts().sort_index())
    
    # 🔥 使用训练集确定基准
    base_year = 2024
    base_month = df.loc[train_mask, '月'].mode()[0] if train_mask.any() else df['月'].mode()[0]
    
    print(f"\n基准设置（基于训练集）:")
    print(f"  基准年: {base_year}")
    print(f"  基准月: {int(base_month)} (训练集频率最高)")
    
    # One-hot编码年份（排除基准年）
    years = sorted(df['年'].dropna().unique())
    year_cols = []
    for year in years:
        if year != base_year:
            col_name = f'年_{int(year)}'
            df[col_name] = (df['年'] == year).astype(int)
            year_cols.append(col_name)
            print(f"  创建: {col_name} ({(df['年']==year).sum()} 个)")
    
    # One-hot编码月份（排除基准月）
    months = sorted(df['月'].dropna().unique())
    month_cols = []
    for month in months:
        if month != base_month:
            col_name = f'月_{int(month)}'
            df[col_name] = (df['月'] == month).astype(int)
            month_cols.append(col_name)
            print(f"  创建: {col_name} ({(df['月']==month).sum()} 个)")
    
    print(f"\n✅ 以 {base_year}年 和 {int(base_month)}月 为基准（不生成虚拟变量）")


# ============ 3. 删除原始列 ============
print("\n" + "-" * 80)
print("删除原始列")
print("-" * 80)

cols_to_drop = []
if direction_col:
    cols_to_drop.append(direction_col)
if time_col:
    cols_to_drop.extend([time_col, '交易时间_dt', '年', '月'])

cols_to_drop = [col for col in cols_to_drop if col in df.columns]

print(f"删除列: {cols_to_drop}")
df = df.drop(columns=cols_to_drop)


# ============ 4. 最终检查 ============
print("\n" + "=" * 80)
print("最终数据")
print("=" * 80)

print(f"数据形状: {df.shape}")

# 列出新增的列
new_cols = [col for col in df.columns if '朝向_' in col or '年_' in col or '月_' in col]
print(f"\n新增列 ({len(new_cols)}个):")
for col in sorted(new_cols):
    print(f"  {col}: {df[col].sum()} 个1")

# 显示前几行
if new_cols:
    print(f"\n数据样例（前10行）:")
    print(df[new_cols].head(10).to_string(index=False))


# ============ 5. 保存 ============
df.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n✅ 完成: {df.shape}")
print(f"✅ 保存: {out_path}")
print(f"✅ 新增变量: {len(new_cols)} 个")

if 'is_train' in df.columns:
    print(f"\n训练集: {(df['is_train']==1).sum():,}")
    print(f"测试集: {(df['is_train']==0).sum():,}")

print("\n" + "=" * 80)
print("✨ 完成")
print("=" * 80)

处理朝向和交易时间
原始数据: (97709, 22)
训练集: 78,184, 测试集: 19,525
朝向列: '朝向'
交易时间列: '交易时间'

--------------------------------------------------------------------------------
处理朝向（东南西北01变量）
--------------------------------------------------------------------------------

原始朝向样例（前20个）:
朝向
南        44134
南 北      16772
东南       10282
东         6211
北         6177
西南        3579
西         3077
东北        1474
西北        1358
东 西        887
东南 南       864
东 南        402
东 东南       286
南 西南       254
东 北        221
南 西        218
东 南 北      135
南 西 北      132
西 北        131
东南 北       112
Name: count, dtype: int64

朝向01变量统计:
  东: 21528 (22.0%)
  南: 77927 (79.8%)
  西: 10384 (10.6%)
  北: 27145 (27.8%)

样例（前10个）:
朝向  朝向_东  朝向_南  朝向_西  朝向_北
 西     0     0     1     0
 南     0     1     0     0
 北     0     0     0     1
 南     0     1     0     0
 南     0     1     0     0
 北     0     0     0     1
 北     0     0     0     1
 北     0     0     0     1
 北     0     0     0     1
西南     0     1     1     0

-----

In [6]:
# -*- coding: utf-8 -*-
"""
租房数据 - 处理楼层和电梯
"""

import pandas as pd
import numpy as np
import re

# ===================== 配置 =====================
path = r"H:\HW\ruc_Class25Q2_train_rent5.csv"
out_path = r"H:\HW\ruc_Class25Q2_train_rent6.csv"
# =================================================

print("处理楼层和电梯")
print("=" * 80)

df = pd.read_csv(path, encoding='utf-8-sig')
print(f"\n原始数据: {df.shape}")

if 'is_train' in df.columns:
    train_mask = df['is_train'] == 1
    print(f"训练集: {train_mask.sum():,}, 测试集: {(~train_mask).sum():,}")
else:
    train_mask = pd.Series([True] * len(df))
    print("⚠️ 未找到is_train列，使用全部数据")

# 找列
floor_col = [col for col in df.columns if '楼层' in col]
elevator_col = [col for col in df.columns if '电梯' in col]

floor_col = floor_col[0] if floor_col else None
elevator_col = elevator_col[0] if elevator_col else None

print(f"楼层列: '{floor_col}'")
print(f"电梯列: '{elevator_col}'")


# ============ 1. 解析楼层 ============
if floor_col:
    print("\n" + "-" * 80)
    print("解析楼层")
    print("-" * 80)
    
    print(f"\n原始楼层样例（前20个）:")
    print(df[floor_col].value_counts().head(20))
    
    def parse_floor(s):
        if pd.isna(s): 
            return np.nan, np.nan
        s = str(s)
        
        # 地下室
        if '地下' in s:
            match = re.search(r'/(\d+)', s)
            return (int(match.group(1)) if match else np.nan), '地下室'
        
        # 中楼层/12层
        m = re.match(r'([低中高])楼层/(\d+)', s)
        if m:
            return int(m.group(2)), m.group(1) + '层'
        
        # 3/32层
        m = re.match(r'(\d+)/(\d+)', s)
        if m:
            cur, tot = int(m.group(1)), int(m.group(2))
            if cur <= tot/3: 
                t = '低层'
            elif cur <= tot*2/3: 
                t = '中层'
            else: 
                t = '高层'
            return tot, t
        
        return np.nan, np.nan
    
    df[['总层数', '楼层类型']] = df[floor_col].apply(lambda x: pd.Series(parse_floor(x)))
    
    print(f"\n楼层类型分布:")
    print(df['楼层类型'].value_counts())
    
    print(f"\n总层数统计:")
    print(f"  均值: {df['总层数'].mean():.1f}")
    print(f"  中位数: {df['总层数'].median():.1f}")
    print(f"  缺失: {df['总层数'].isna().sum()}")
    
    # 🔥 用训练集众数填补楼层类型
    floor_type_mode = df.loc[train_mask, '楼层类型'].mode()[0] if train_mask.any() else df['楼层类型'].mode()[0]
    floor_type_missing = df['楼层类型'].isna().sum()
    df['楼层类型'].fillna(floor_type_mode, inplace=True)
    
    print(f"\n✅ 用训练集众数'{floor_type_mode}'填充{floor_type_missing}个缺失值")
    
    # 楼层one-hot（低层为基准）
    df['楼层_中层'] = (df['楼层类型']=='中层').astype(int)
    df['楼层_高层'] = (df['楼层类型']=='高层').astype(int)
    df['楼层_地下室'] = (df['楼层类型']=='地下室').astype(int)
    
    print(f"\n楼层one-hot统计（低层为基准）:")
    print(f"  楼层_中层: {df['楼层_中层'].sum()} ({df['楼层_中层'].sum()/len(df)*100:.1f}%)")
    print(f"  楼层_高层: {df['楼层_高层'].sum()} ({df['楼层_高层'].sum()/len(df)*100:.1f}%)")
    print(f"  楼层_地下室: {df['楼层_地下室'].sum()} ({df['楼层_地下室'].sum()/len(df)*100:.1f}%)")


# ============ 2. 解析电梯 ============
if elevator_col:
    print("\n" + "-" * 80)
    print("解析电梯")
    print("-" * 80)
    
    print(f"\n原始电梯样例:")
    print(df[elevator_col].value_counts())
    
    def parse_elevator(s):
        if pd.isna(s): 
            return np.nan
        s = str(s).lower()
        if '有' in s: 
            return 1
        if '无' in s: 
            return 0
        return np.nan
    
    df['电梯_有'] = df[elevator_col].apply(parse_elevator)
    
    print(f"\n解析后:")
    print(f"  有电梯: {(df['电梯_有']==1).sum()}")
    print(f"  无电梯: {(df['电梯_有']==0).sum()}")
    print(f"  缺失: {df['电梯_有'].isna().sum()}")
    
    # 用总层数填补电梯（>6层有电梯）
    if '总层数' in df.columns:
        mask = df['电梯_有'].isna() & df['总层数'].notna()
        filled_by_floor = mask.sum()
        df.loc[mask, '电梯_有'] = (df.loc[mask, '总层数'] > 6).astype(int)
        print(f"\n✅ 用总层数(>6层)填充{filled_by_floor}个缺失值")
    
    # 🔥 剩余用训练集众数填充
    elevator_mode = df.loc[train_mask, '电梯_有'].mode()[0] if train_mask.any() else df['电梯_有'].mode()[0]
    elevator_remaining = df['电梯_有'].isna().sum()
    df['电梯_有'].fillna(elevator_mode, inplace=True)
    df['电梯_有'] = df['电梯_有'].astype(int)
    
    print(f"✅ 用训练集众数{int(elevator_mode)}填充剩余{elevator_remaining}个缺失值")
    
    print(f"\n最终电梯统计:")
    print(f"  有电梯: {(df['电梯_有']==1).sum()} ({(df['电梯_有']==1).sum()/len(df)*100:.1f}%)")
    print(f"  无电梯: {(df['电梯_有']==0).sum()} ({(df['电梯_有']==0).sum()/len(df)*100:.1f}%)")


# ============ 3. 删除原始列 ============
print("\n" + "-" * 80)
print("删除原始列")
print("-" * 80)

cols_to_drop = []
if floor_col:
    cols_to_drop.extend([floor_col, '总层数', '楼层类型'])
if elevator_col:
    cols_to_drop.append(elevator_col)

cols_to_drop = [col for col in cols_to_drop if col in df.columns]

print(f"删除列: {cols_to_drop}")
df = df.drop(columns=cols_to_drop)


# ============ 4. 最终检查 ============
print("\n" + "=" * 80)
print("最终数据")
print("=" * 80)

print(f"数据形状: {df.shape}")

# 列出新增的列
new_cols = [col for col in df.columns if '楼层_' in col or '电梯_' in col]
print(f"\n新增列 ({len(new_cols)}个):")
for col in sorted(new_cols):
    print(f"  {col}: {df[col].sum()} 个1 ({df[col].sum()/len(df)*100:.1f}%)")

# 显示前几行
if new_cols:
    print(f"\n数据样例（前10行）:")
    print(df[new_cols].head(10).to_string(index=False))


# ============ 5. 保存 ============
df.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n✅ 完成: {df.shape}")
print(f"✅ 保存: {out_path}")
print(f"✅ 新增变量: {', '.join(new_cols)}")

if 'is_train' in df.columns:
    print(f"\n训练集: {(df['is_train']==1).sum():,}")
    print(f"测试集: {(df['is_train']==0).sum():,}")

print("\n" + "=" * 80)
print("✨ 完成")
print("=" * 80)

处理楼层和电梯

原始数据: (97709, 36)
训练集: 78,184, 测试集: 19,525
楼层列: '楼层'
电梯列: '电梯'

--------------------------------------------------------------------------------
解析楼层
--------------------------------------------------------------------------------

原始楼层样例（前20个）:
楼层
高楼层/6层     5004
中楼层/6层     3482
低楼层/6层     2586
中楼层/18层    2110
高楼层/18层    2100
低楼层/18层    2088
中楼层/32层    1724
高楼层/33层    1657
中楼层/7层     1651
低楼层/33层    1580
中楼层/5层     1561
中楼层/33层    1528
高楼层/32层    1456
中楼层/11层    1404
低楼层/32层    1327
高楼层/7层     1306
中楼层/27层    1163
中楼层/8层     1137
高楼层/27层    1057
中楼层/31层    1035
Name: count, dtype: int64

楼层类型分布:
楼层类型
中层     36745
高层     33034
低层     27666
地下室      259
Name: count, dtype: int64

总层数统计:
  均值: 19.8
  中位数: 19.0
  缺失: 12

✅ 用训练集众数'中层'填充5个缺失值

楼层one-hot统计（低层为基准）:
  楼层_中层: 36750 (37.6%)
  楼层_高层: 33034 (33.8%)
  楼层_地下室: 259 (0.3%)

--------------------------------------------------------------------------------
解析电梯
--------------------------------------------------------------------

C:\Users\Administrator\AppData\Local\Temp\ipykernel_21440\2837090287.py:90: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['楼层类型'].fillna(floor_type_mode, inplace=True)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_21440\2837090287.py:141: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves 


✅ 完成: (97709, 38)
✅ 保存: H:\HW\ruc_Class25Q2_train_rent6.csv
✅ 新增变量: 楼层_中层, 楼层_高层, 楼层_地下室, 电梯_有

训练集: 78,184
测试集: 19,525

✨ 完成


In [7]:
# -*- coding: utf-8 -*-
"""
租房数据 - 处理付款/租赁方式和数值列（按城市分组处理）
"""

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize
import re

# ===================== 配置 =====================
path = r"H:\HW\ruc_Class25Q2_train_rent6.csv"
out_path = r"H:\HW\ruc_Class25Q2_train_rent7.csv"
# =================================================

print("=" * 80)
print("处理付款/租赁方式和数值列（按城市分组）")
print("=" * 80)

df = pd.read_csv(path, encoding='utf-8-sig')
print(f"\n原始数据: {df.shape}")

if 'is_train' in df.columns:
    train_mask = df['is_train'] == 1
    print(f"训练集: {train_mask.sum():,}, 测试集: {(~train_mask).sum():,}")
else:
    train_mask = pd.Series([True] * len(df))

# 🔥 找到城市列
city_column = None
for col in df.columns:
    if '城市' in col or 'city' in col.lower():
        city_column = col
        break

if city_column is None:
    print("⚠️ 未找到城市列，将使用全局众数")
    use_city_mode = False
else:
    print(f"✅ 找到城市列: {city_column}")
    use_city_mode = True
    city_list = df[city_column].unique()
    print(f"   共{len(city_list)}个城市: {sorted(city_list)}")


# ============ 1. 付款方式和租赁方式 one-hot（按城市分组）============
print("\n" + "-" * 80)
print("Step 1: 处理付款/租赁方式 (按城市分组 one-hot编码)")

payment_col = [col for col in df.columns if '付款' in col]
lease_col = [col for col in df.columns if '租赁' in col]

payment_col = payment_col[0] if payment_col else None
lease_col = lease_col[0] if lease_col else None

new_dummy_cols = []

if payment_col:
    print(f"\n  付款方式原始分布:")
    print(df[payment_col].value_counts())
    
    # 🔥 处理异常值（URL等）
    def is_invalid_payment(x):
        if pd.isna(x):
            return True
        x_str = str(x).lower()
        return 'http' in x_str or 'ljcdn' in x_str or 'image' in x_str or '.com' in x_str
    
    invalid_mask = df[payment_col].apply(is_invalid_payment)
    invalid_count = invalid_mask.sum()
    
    if invalid_count > 0:
        print(f"  发现{invalid_count}个异常值（URL等），替换为缺失值")
        df.loc[invalid_mask, payment_col] = np.nan
    
    # 🔥 按城市分组填充
    if use_city_mode:
        print(f"\n  按城市分组填充付款方式:")
        print(f"  {'城市':<10} | {'训练集众数':<20} | {'填充训练集':>10} | {'填充测试集':>10}")
        print("  " + "-" * 65)
        
        for city in city_list:
            city_mask = df[city_column] == city
            city_train_mask = city_mask & train_mask
            city_test_mask = city_mask & ~train_mask
            
            # 计算该城市训练集的众数
            city_train_values = df.loc[city_train_mask, payment_col]
            if city_train_values.notna().sum() > 0:
                city_mode = city_train_values.mode()[0]
            else:
                # 如果该城市训练集没有有效值，使用全局训练集众数
                city_mode = df.loc[train_mask, payment_col].mode()[0]
            
            # 填充该城市训练集的缺失值
            train_missing = df.loc[city_train_mask, payment_col].isna().sum()
            df.loc[city_train_mask & df[payment_col].isna(), payment_col] = city_mode
            
            # 填充该城市测试集的缺失值
            test_missing = df.loc[city_test_mask, payment_col].isna().sum()
            df.loc[city_test_mask & df[payment_col].isna(), payment_col] = city_mode
            
            print(f"  城市{city:<8} | {city_mode:<20} | {train_missing:>10} | {test_missing:>10}")
    else:
        # 使用全局众数
        train_mode = df.loc[train_mask, payment_col].mode()[0]
        missing_count = df[payment_col].isna().sum()
        df[payment_col].fillna(train_mode, inplace=True)
        print(f"  用训练集众数'{train_mode}'填充{missing_count}个缺失值")
    
    # One-hot编码
    train_categories = df.loc[train_mask, payment_col].value_counts().index.tolist()
    payment_dummies = pd.get_dummies(df[payment_col], prefix='付款方式', dtype=int)
    
    base_category = f'付款方式_{train_categories[0]}'
    if base_category in payment_dummies.columns:
        payment_dummies.drop(columns=[base_category], inplace=True)
    
    df = pd.concat([df, payment_dummies], axis=1)
    df.drop(columns=[payment_col], inplace=True)
    new_dummy_cols.extend(list(payment_dummies.columns))
    print(f"\n  付款方式: 基准={train_categories[0]}, 新增{len(payment_dummies.columns)}列")

if lease_col:
    print(f"\n  租赁方式原始分布:")
    print(df[lease_col].value_counts())
    
    # 检查租赁方式的异常值
    def is_invalid_lease(x):
        if pd.isna(x):
            return True
        x_str = str(x).lower()
        return 'http' in x_str or 'ljcdn' in x_str or 'image' in x_str or '.com' in x_str
    
    invalid_mask = df[lease_col].apply(is_invalid_lease)
    invalid_count = invalid_mask.sum()
    
    if invalid_count > 0:
        print(f"  租赁方式发现{invalid_count}个异常值，替换为缺失值")
        df.loc[invalid_mask, lease_col] = np.nan
    
    # 🔥 按城市分组填充
    if use_city_mode:
        print(f"\n  按城市分组填充租赁方式:")
        print(f"  {'城市':<10} | {'训练集众数':<20} | {'填充训练集':>10} | {'填充测试集':>10}")
        print("  " + "-" * 65)
        
        for city in city_list:
            city_mask = df[city_column] == city
            city_train_mask = city_mask & train_mask
            city_test_mask = city_mask & ~train_mask
            
            # 计算该城市训练集的众数
            city_train_values = df.loc[city_train_mask, lease_col]
            if city_train_values.notna().sum() > 0:
                city_mode = city_train_values.mode()[0]
            else:
                city_mode = df.loc[train_mask, lease_col].mode()[0]
            
            # 填充该城市训练集的缺失值
            train_missing = df.loc[city_train_mask, lease_col].isna().sum()
            df.loc[city_train_mask & df[lease_col].isna(), lease_col] = city_mode
            
            # 填充该城市测试集的缺失值
            test_missing = df.loc[city_test_mask, lease_col].isna().sum()
            df.loc[city_test_mask & df[lease_col].isna(), lease_col] = city_mode
            
            print(f"  城市{city:<8} | {city_mode:<20} | {train_missing:>10} | {test_missing:>10}")
    else:
        # 使用全局众数
        train_mode = df.loc[train_mask, lease_col].mode()[0]
        missing_count = df[lease_col].isna().sum()
        df[lease_col].fillna(train_mode, inplace=True)
        print(f"  用训练集众数'{train_mode}'填充{missing_count}个缺失值")
    
    # One-hot编码
    train_categories = df.loc[train_mask, lease_col].value_counts().index.tolist()
    lease_dummies = pd.get_dummies(df[lease_col], prefix='租赁方式', dtype=int)
    
    base_category = f'租赁方式_{train_categories[0]}'
    if base_category in lease_dummies.columns:
        lease_dummies.drop(columns=[base_category], inplace=True)
    
    df = pd.concat([df, lease_dummies], axis=1)
    df.drop(columns=[lease_col], inplace=True)
    new_dummy_cols.extend(list(lease_dummies.columns))
    print(f"\n  租赁方式: 基准={train_categories[0]}, 新增{len(lease_dummies.columns)}列")


# ============ 2. 物业费解析 ============
print("\n" + "-" * 80)
print("Step 2: 解析物业费")

property_col = [col for col in df.columns if '物业费' in col]
property_col = property_col[0] if property_col else None

if property_col:
    def parse_property_fee(x):
        if pd.isna(x): return np.nan
        x = str(x)
        numbers = re.findall(r'\d+\.?\d*', x)
        if not numbers: return np.nan
        numbers = [float(n) for n in numbers]
        return np.mean(numbers) if len(numbers) >= 2 else numbers[0]
    
    df[property_col] = df[property_col].apply(parse_property_fee)
    print(f"  {property_col}: 解析完成")


# ============ 3. 数值列处理（按城市分组）============
print("\n" + "-" * 80)
print("Step 3: 处理数值列 (按城市分组，训练集Winsorize, 训练集中位数填补)")

numeric_keywords = ['房屋总数', '楼栋总数', '绿化率', '容积率', '物业费']
found_cols = []
for keyword in numeric_keywords:
    for col in df.columns:
        if keyword in col and col not in found_cols:
            found_cols.append(col)
            break

print(f"  处理列: {found_cols}")

for col in found_cols:
    print(f"\n  处理列: {col}")
    
    # 去单位
    if '物业费' not in col:
        def clean_numeric(x):
            if pd.isna(x): return np.nan
            x = re.sub(r'[^\d.]', '', str(x))
            return float(x) if x else np.nan
        df[col] = df[col].apply(clean_numeric)
    
    # 🔥 按城市分组处理
    if use_city_mode:
        print(f"    {'城市':<10} | {'中位数':>10} | {'填充训练':>10} | {'填充测试':>10}")
        print("    " + "-" * 50)
        
        for city in city_list:
            city_mask = df[city_column] == city
            city_train_mask = city_mask & train_mask
            city_test_mask = city_mask & ~train_mask
            
            # 仅该城市训练集Winsorize
            train_valid_mask = city_train_mask & df[col].notna()
            if train_valid_mask.sum() > 0:
                train_data = df.loc[train_valid_mask, col].values
                df.loc[train_valid_mask, col] = winsorize(train_data, limits=(0.01, 0.01))
            
            # 计算该城市训练集中位数
            city_train_values = df.loc[city_train_mask, col]
            if city_train_values.notna().sum() > 0:
                city_median = city_train_values.median()
            else:
                # 如果该城市训练集没有有效值，使用全局训练集中位数
                city_median = df.loc[train_mask, col].median()
            
            # 填充该城市训练集和测试集的缺失值
            train_missing = df.loc[city_train_mask, col].isna().sum()
            df.loc[city_train_mask & df[col].isna(), col] = city_median
            
            test_missing = df.loc[city_test_mask, col].isna().sum()
            df.loc[city_test_mask & df[col].isna(), col] = city_median
            
            print(f"    城市{city:<8} | {city_median:>10.2f} | {train_missing:>10} | {test_missing:>10}")
    else:
        # 全局处理
        train_valid_mask = train_mask & df[col].notna()
        if train_valid_mask.sum() > 0:
            train_data = df.loc[train_valid_mask, col].values
            df.loc[train_valid_mask, col] = winsorize(train_data, limits=(0.01, 0.01))
        
        train_median = df.loc[train_mask, col].median()
        total_missing = df[col].isna().sum()
        df[col].fillna(train_median, inplace=True)
        print(f"    全局中位数={train_median:.2f}, 填补{total_missing}个缺失")


# ============ 4. 保存 ============
print("\n" + "=" * 80)
print("完成")
print("=" * 80)

df.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n最终数据: {df.shape}")
if new_dummy_cols:
    print(f"新增类别变量: {new_dummy_cols}")
print(f"✅ 保存: {out_path}")

if 'is_train' in df.columns:
    print(f"\n训练集: {(df['is_train']==1).sum():,}")
    print(f"测试集: {(df['is_train']==0).sum():,}")
print("=" * 80)

处理付款/租赁方式和数值列（按城市分组）

原始数据: (97709, 38)
训练集: 78,184, 测试集: 19,525
✅ 找到城市列: 城市
   共12个城市: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]

--------------------------------------------------------------------------------
Step 1: 处理付款/租赁方式 (按城市分组 one-hot编码)

  付款方式原始分布:
付款方式
季付价                               53389
月付价                               20913
半年付价                               4194
年付价                                 744
双月付价                                 44
https://img.ljcdn.com/usercent        3
https://image1.ljcdn.com/rent-        2
Name: count, dtype: int64
  发现18425个异常值（URL等），替换为缺失值

  按城市分组填充付款方式:
  城市         | 训练集众数                |      填充训练集 |      填充测试集
  -----------------------------------------------------------------
  城市0        | 季付价                  |        685 |        172
  城市1        | 季付价                  |       4428 |       1069
  城市2        |

In [8]:
# -*- coding: utf-8 -*-
"""
计算月租金 - 根据付款方式one-hot列
"""

import pandas as pd
import numpy as np

# ===================== 配置 =====================
path_in = r"H:\HW\ruc_Class25Q2_train_rent7.csv"   # 输入文件（已one-hot）
path_out = r"H:\HW\ruc_Class25Q2_train_rent8.csv"  # 输出文件（新增月租列）
# =================================================

print("=" * 80)
print("计算月租金 - 根据付款方式one-hot列")
print("=" * 80)

# 读取数据
df = pd.read_csv(path_in, encoding='utf-8-sig')
print(f"\n原始数据: {df.shape}")

if 'Price' in df.columns:
    print(f"\n原始Price统计:")
    print(f"  均值:   {df['Price'].mean():>12,.0f} 元")
    print(f"  中位数: {df['Price'].median():>12,.0f} 元")
    print(f"  范围:   [{df['Price'].min():>10,.0f}, {df['Price'].max():>10,.0f}]")
    
    # 付款方式one-hot列与转换系数的映射
    payment_mapping = {
        '付款方式_月付价': 1,
        '付款方式_双月付价': 2,
        '付款方式_季付价': 3,
        '付款方式_半年付价': 6,
        '付款方式_年付价': 12
    }
    
    # 初始化月租列（先复制Price）
    df['月租'] = df['Price'].copy()
    
    # 统计转换情况
    print(f"\n付款方式转换统计:")
    print(f"  {'付款方式列':<20} | {'样本数':>8} | {'除数':>5} | {'原Price均值':>12} | {'月租均值':>12}")
    print("  " + "-" * 78)
    
    found_columns = []
    
    for col_name, divisor in payment_mapping.items():
        if col_name in df.columns:
            found_columns.append(col_name)
            # 找到该付款方式=1的行
            mask = df[col_name] == 1
            count = mask.sum()
            
            if count > 0:
                avg_before = df.loc[mask, 'Price'].mean()
                
                # 计算月租
                df.loc[mask, '月租'] = df.loc[mask, 'Price'] / divisor
                
                avg_after = df.loc[mask, '月租'].mean()
                
                print(f"  {col_name:<20} | {count:>8,} | {divisor:>5} | "
                      f"{avg_before:>12,.0f} | {avg_after:>12,.0f}")
    
    if not found_columns:
        print("  ⚠️ 未找到任何付款方式one-hot列")
    
    # 检查是否有基准类别（未编码的付款方式）
    total_matched = sum((df[col] == 1).sum() for col in found_columns if col in df.columns)
    total_samples = len(df)
    unmatched = total_samples - total_matched
    
    if unmatched > 0:
        print(f"\n  ℹ️ 有{unmatched:,}个样本属于基准类别（默认月租=Price）")
    
    print(f"\n月租列统计:")
    print(f"  均值:   {df['月租'].mean():>12,.0f} 元")
    print(f"  中位数: {df['月租'].median():>12,.0f} 元")
    print(f"  范围:   [{df['月租'].min():>10,.0f}, {df['月租'].max():>10,.0f}]")
    
    print(f"\n✅ 新增'月租'列")
else:
    print("⚠️ 未找到Price列，无法计算月租")

# 保存结果
df.to_csv(path_out, index=False, encoding='utf-8-sig')
print(f"\n" + "=" * 80)
print(f"✅ 保存成功: {path_out}")
print(f"   最终数据: {df.shape}")
if 'is_train' in df.columns:
    print(f"   训练集: {(df['is_train']==1).sum():,}")
    print(f"   测试集: {(df['is_train']==0).sum():,}")
print("=" * 80)

计算月租金 - 根据付款方式one-hot列

原始数据: (97709, 41)

原始Price统计:
  均值:        586,986 元
  中位数:      400,489 元
  范围:   [    17,938, 15,404,193]

付款方式转换统计:
  付款方式列                |      样本数 |    除数 |     原Price均值 |         月租均值
  ------------------------------------------------------------------------------
  付款方式_月付价             |   24,999 |     1 |      620,348 |      620,348
  付款方式_双月付价            |       44 |     2 |      653,928 |      326,964
  付款方式_半年付价            |    7,508 |     6 |      324,053 |       54,009
  付款方式_年付价             |    1,566 |    12 |      309,720 |       25,810

  ℹ️ 有63,592个样本属于基准类别（默认月租=Price）

月租列统计:
  均值:        561,538 元
  中位数:      388,658 元
  范围:   [     2,903, 15,404,193]

✅ 新增'月租'列

✅ 保存成功: H:\HW\ruc_Class25Q2_train_rent8.csv
   最终数据: (97709, 42)
   训练集: 78,184
   测试集: 19,525


In [26]:
# -*- coding: utf-8 -*-
"""
租房数据 - 处理配套设施 + 清理面积单位 + 删除建筑年代
"""

import pandas as pd
import numpy as np
from collections import Counter
import re

# ===================== 配置 =====================
path = r"H:\HW\ruc_Class25Q2_train_rent8.csv"
out_path = r"H:\HW\ruc_Class25Q2_train_rent9.csv"
# =================================================

print("=" * 80)
print("处理面积单位 + 配套设施 + 删除建筑年代")
print("=" * 80)

df = pd.read_csv(path, encoding='utf-8-sig')
print(f"\n原始数据: {df.shape}")

if 'is_train' in df.columns:
    train_mask = df['is_train'] == 1
    test_mask = df['is_train'] == 0
    print(f"训练集: {train_mask.sum():,}, 测试集: {test_mask.sum():,}")
else:
    train_mask = pd.Series([True] * len(df))
    test_mask = pd.Series([False] * len(df))


# ============ 0. 清理面积单位 ============
print("\n" + "-" * 80)
print("Step 0: 清理面积单位（去除m²）")

# 找面积列
area_col = None
for col in df.columns:
    if '面积' in col or col == '面积':
        area_col = col
        break

if area_col:
    print(f"  找到面积列: '{area_col}'")
    
    # 查看原始数据示例
    sample_before = df[area_col].head(5).tolist()
    print(f"  原始示例: {sample_before}")
    
    # 如果已经是数值型，则跳过
    if pd.api.types.is_numeric_dtype(df[area_col]):
        print(f"  '{area_col}' 已是数值型，无需处理")
    else:
        # 转换为字符串，去除单位
        df[area_col] = df[area_col].astype(str)
        
        # 去除常见单位：m²、㎡、平米、平方米、平、方
        df[area_col] = df[area_col].str.replace(r'[m²㎡平米方]+', '', regex=True)
        df[area_col] = df[area_col].str.replace('平方米', '', regex=False)
        df[area_col] = df[area_col].str.replace('平方', '', regex=False)
        
        # 去除空格
        df[area_col] = df[area_col].str.strip()
        
        # 转换为数值型
        df[area_col] = pd.to_numeric(df[area_col], errors='coerce')
        
        # 统计转换结果
        missing_count = df[area_col].isna().sum()
        if missing_count > 0:
            print(f"  ⚠️ 转换后有 {missing_count} 个缺失值")
        
        sample_after = df[area_col].head(5).tolist()
        print(f"  转换后示例: {sample_after}")
        print(f"  ✅ '{area_col}' 已转换为数值型")
        print(f"     均值: {df[area_col].mean():.2f}, 中位数: {df[area_col].median():.2f}")
else:
    print(f"  ⚠️ 未找到面积列")


# ============ 1. 找配套设施列 ============
facility_col = df.columns[0]
for col in df.columns:
    if '配套' in col or '设施' in col:
        facility_col = col
        break

print(f"\n配套设施列: '{facility_col}'")


# ============ 2. 数据清洗 ============
print("\n" + "-" * 80)
print("Step 1: 数据清洗")

# 天然气相关变体统一
df[facility_col] = df[facility_col].astype(str).str.replace('天然气气', '天然气', regex=False)
df[facility_col] = df[facility_col].astype(str).str.replace(r'(?<![气])天然(?![气])', '天然气', regex=True)
print(f"  统一'天然'、'天然气气' → '天然气'")


# ============ 3. 基于训练集统计物品频率 ============
print("\n" + "-" * 80)
print("Step 2: 基于训练集统计物品频率")

# 🔥 仅从训练集提取物品
train_data = df.loc[train_mask, facility_col].dropna()
all_items = []
for val in train_data:
    # 按常见分隔符拆分
    items = re.split(r'[、，,\s/]+', str(val))
    all_items.extend([item.strip() for item in items if item.strip() and item.strip() != 'nan'])

# 统计物品频率
item_counter = Counter(all_items)
item_freq = sorted(item_counter.items(), key=lambda x: x[1], reverse=True)

print(f"  训练集识别出 {len(item_counter)} 种物品")
print(f"  Top 20:")
for item, count in item_freq[:20]:
    print(f"    {item}: {count}次")


# ============ 4. 计算训练集覆盖率 ============
print("\n" + "-" * 80)
print("Step 3: 计算训练集覆盖率")

train_total = train_mask.sum()
item_coverage = {}

for item, _ in item_freq[:20]:
    count = df.loc[train_mask, facility_col].astype(str).str.contains(item, na=False, regex=False).sum()
    ratio = count / train_total * 100
    item_coverage[item] = (count, ratio)

# 找出高频物品（>=50%）
common_items = [item for item, (cnt, ratio) in item_coverage.items() if ratio >= 50]
print(f"  高频物品(>=50%): {common_items}")


# ============ 5. 填补缺失（用训练集高频物品）============
print("\n" + "-" * 80)
print("Step 4: 填补缺失（训练集+测试集）")

train_missing = (train_mask & df[facility_col].isna()).sum()
test_missing = (test_mask & df[facility_col].isna()).sum()
total_missing = df[facility_col].isna().sum()

if total_missing > 0 and common_items:
    fill_text = '、'.join(common_items)
    df.loc[df[facility_col].isna(), facility_col] = fill_text
    print(f"  填补内容: {fill_text}")
    print(f"  填补数量: 训练集{train_missing}个, 测试集{test_missing}个, 共{total_missing}个")
else:
    print(f"  无需填补")


# ============ 6. 创建01变量（前20高频物品）============
print("\n" + "-" * 80)
print("Step 5: 创建01变量（基于训练集Top 20）")

items_to_encode = [item for item, _ in item_freq[:20]]
new_cols = []

for item in items_to_encode:
    col_name = f'设施_{item}'
    df[col_name] = df[facility_col].astype(str).str.contains(item, na=False, regex=False).astype(int)
    new_cols.append(col_name)
    
    train_count = df.loc[train_mask, col_name].sum()
    train_ratio = train_count / train_mask.sum() * 100
    print(f"  {col_name}: 训练集{train_count}个 ({train_ratio:.1f}%)")

# 删除原始列
df.drop(columns=[facility_col], inplace=True)


# ============ 7. 删除建筑年代列 ============
print("\n" + "-" * 80)
print("Step 6: 删除建筑年代列")

# 查找建筑年代相关列
year_cols = [col for col in df.columns if '建筑年代' in col or col == '建筑年代']

if year_cols:
    print(f"  找到建筑年代相关列: {year_cols}")
    df.drop(columns=year_cols, inplace=True)
    print(f"  ✅ 已删除 {len(year_cols)} 个建筑年代列")
else:
    print(f"  ⚠️ 未找到建筑年代列")


# ============ 8. 保存 ============
print("\n" + "=" * 80)
print("完成")
print("=" * 80)

df.to_csv(out_path, index=False, encoding='utf-8-sig')

print(f"\n最终数据: {df.shape}")
if area_col:
    print(f"✅ 面积列已清理单位")
print(f"✅ 新增变量: {len(new_cols)}个设施01变量")
print(f"✅ 删除列: {facility_col}")
if year_cols:
    print(f"✅ 删除列: {', '.join(year_cols)}")
print(f"✅ 保存: {out_path}")

if 'is_train' in df.columns:
    print(f"\n训练集: {(df['is_train']==1).sum():,}")
    print(f"测试集: {(df['is_train']==0).sum():,}")
print("=" * 80)

处理面积单位 + 配套设施 + 删除建筑年代

原始数据: (97709, 42)
训练集: 78,184, 测试集: 19,525

--------------------------------------------------------------------------------
Step 0: 清理面积单位（去除m²）
  找到面积列: '面积'
  原始示例: [36.42, 41.0, 37.36, 55.42, 49.3]
  '面积' 已是数值型，无需处理

配套设施列: '配套设施'

--------------------------------------------------------------------------------
Step 1: 数据清洗
  统一'天然'、'天然气气' → '天然气'

--------------------------------------------------------------------------------
Step 2: 基于训练集统计物品频率
  训练集识别出 10 种物品
  Top 20:
    床: 49406次
    衣柜: 44679次
    空调: 43875次
    洗衣机: 42754次
    热水器: 42466次
    冰箱: 38549次
    天然气: 32984次
    电视: 28946次
    暖气: 14622次
    宽带: 12594次

--------------------------------------------------------------------------------
Step 3: 计算训练集覆盖率
  高频物品(>=50%): ['床', '衣柜', '空调', '洗衣机', '热水器']

--------------------------------------------------------------------------------
Step 4: 填补缺失（训练集+测试集）
  无需填补

--------------------------------------------------------------------------------
Ste

In [34]:
# -*- coding: utf-8 -*-
"""
基于训练集聚类，测试集找最近训练样本复制特征
"""

import pandas as pd
import numpy as np
from pathlib import Path
import warnings

# ===================== 配置区 =====================
path_in = r"H:\HW\ruc_Class25Q2_train_rent9.csv"

# 固定列名
BLOCK_COL  = "板块"
LON_COL    = "lon"
LAT_COL    = "lat"

# 大小样本阈值
min_count_big_block = 100

# 小样本簇最小样本数
min_samples_per_small_cluster = 100

# 融入判断参数
max_merge_to_big_km = 50
distance_ratio_threshold = 1.2

# 小样本聚类参数
target_small_samples_per_cluster = 200
k_small_min, k_small_max = 5, 100
random_state = 42

# 输出文件名后缀
out_suffix = "1.csv"
# =================================================


# ===================== 工具函数 =====================
def read_csv_smart(path):
    for e in ["utf-8-sig", "gbk", "utf-8"]:
        try:
            return pd.read_csv(path, encoding=e), e
        except Exception:
            pass
    raise RuntimeError("无法读取CSV，请检查路径或编码。")

def lonlat_to_mercator(lon_deg: np.ndarray, lat_deg: np.ndarray):
    R = 6378137.0
    MAX_LAT = 85.05112878
    lon = np.asarray(lon_deg, dtype=float)
    lat = np.asarray(lat_deg, dtype=float)
    lat = np.clip(lat, -MAX_LAT, MAX_LAT)
    lon_rad = np.deg2rad(lon)
    lat_rad = np.deg2rad(lat)
    x = R * lon_rad
    y = R * np.log(np.tan(np.pi / 4.0 + lat_rad / 2.0))
    return np.column_stack([x, y])

def haversine_km(lon1, lat1, lon2, lat2):
    R = 6371.0088
    lon1 = np.radians(lon1); lat1 = np.radians(lat1)
    lon2 = np.radians(lon2); lat2 = np.radians(lat2)
    dlon = lon2 - lon1; dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2.0)**2
    c = 2*np.arcsin(np.sqrt(a))
    return R * c

def choose_k_small(total_samples, target_per_cluster, kmin, kmax, n_blocks, min_per_cluster):
    if total_samples <= 0 or n_blocks <= 0:
        return 0
    k = int(np.ceil(total_samples / float(target_per_cluster)))
    k_max_allowed = int(np.floor(total_samples / float(min_per_cluster)))
    if k_max_allowed < 1:
        return 0
    k = min(k, k_max_allowed)
    k = max(kmin, min(k, kmax))
    k = min(k, n_blocks)
    return k

def one_hot_drop_baseline(df, col, prefix):
    counts = df[col].value_counts(dropna=False)
    if counts.empty:
        return df, None, []
    baseline = counts.index[0]
    dummies = pd.get_dummies(df[col], prefix=prefix, dtype=int)
    base_col = f"{prefix}_{baseline}"
    if base_col in dummies.columns:
        dummies.drop(columns=[base_col], inplace=True)
    df = pd.concat([df, dummies], axis=1)
    return df, baseline, list(dummies.columns)


# ===================== 主流程 =====================
def main():
    print("=" * 60)
    print("训练集聚类 + 测试集最近邻分配（每簇≥100样本）")
    print("=" * 60)
    
    # 1. 读取数据
    df, enc = read_csv_smart(path_in)
    for c in [BLOCK_COL, LON_COL, LAT_COL]:
        if c not in df.columns:
            raise KeyError(f"缺少必需列：{c}")
    
    if 'is_train' not in df.columns:
        raise KeyError("缺少is_train列")
    
    # 🔥 1.5 清理面积列（去除单位）
    area_cols = [col for col in df.columns if '面积' in col]
    if area_cols:
        print(f"\n[清理面积列] 找到 {len(area_cols)} 个面积列: {area_cols}")
        for col in area_cols:
            if df[col].dtype == 'object':
                print(f"  处理: {col}")
                # 去除常见单位
                df[col] = df[col].astype(str).str.replace(r'[m²㎡平米方]+', '', regex=True)
                df[col] = df[col].str.replace('平方米', '', regex=False)
                df[col] = df[col].str.replace('平方', '', regex=False)
                df[col] = df[col].str.strip()
                
                # 转换为数值
                df[col] = pd.to_numeric(df[col], errors='coerce')
                
                # 填充缺失值
                missing_count = df[col].isna().sum()
                if missing_count > 0:
                    median_val = df[col].median()
                    df[col].fillna(median_val, inplace=True)
                    print(f"    ✅ 转换完成，填充了 {missing_count} 个缺失值（中位数={median_val:.2f}）")
                else:
                    print(f"    ✅ 转换完成")
            else:
                print(f"  {col} 已是数值类型，跳过")
    
    train_mask = df['is_train'] == 1
    test_mask = df['is_train'] == 0
    
    print(f"\n总样本: {len(df)} | 训练: {train_mask.sum()} | 测试: {test_mask.sum()}")
    
    # 后续代码保持不变...
    # 2. 训练集板块统计
    df_train = df[train_mask].copy()
    g = df_train[df_train[BLOCK_COL].notna()].groupby(BLOCK_COL)
    blk_stats = g.agg(
        blk_count=(BLOCK_COL, "size"),
        lon_med=(LON_COL, "median"),
        lat_med=(LAT_COL, "median")
    ).reset_index()
    
    valid_mask = blk_stats["lon_med"].notna() & blk_stats["lat_med"].notna()
    blk_valid = blk_stats[valid_mask].copy()
    
    if blk_valid.empty:
        raise RuntimeError("训练集所有板块均缺少中心点")
    
    # 3. 划分大小板块
    blk_valid["is_big"] = blk_valid["blk_count"] >= min_count_big_block
    big_blocks = blk_valid[blk_valid["is_big"]].copy()
    small_blocks = blk_valid[~blk_valid["is_big"]].copy()
    
    n_big = len(big_blocks)
    n_small = len(small_blocks)
    
    print(f"\n[训练集板块] 大样本: {n_big}个 | 小样本: {n_small}个")
    
    # 初始化映射
    block_to_cluster = {}
    cluster_centers = {}
    next_cluster_id = 0
    
    # 4. 大样本独立成簇
    for idx, row in big_blocks.iterrows():
        block_name = row[BLOCK_COL]
        block_to_cluster[block_name] = next_cluster_id
        cluster_centers[next_cluster_id] = (row["lon_med"], row["lat_med"])
        next_cluster_id += 1
    
    # 5. 小样本智能分组
    if n_small > 0 and n_big > 0:
        big_lons = big_blocks["lon_med"].values
        big_lats = big_blocks["lat_med"].values
        small_lons = small_blocks["lon_med"].values
        small_lats = small_blocks["lat_med"].values
        
        dist_to_big = np.zeros((n_small, n_big))
        for i, (slon, slat) in enumerate(zip(small_lons, small_lats)):
            dist_to_big[i, :] = haversine_km(slon, slat, big_lons, big_lats)
        
        dist_small = np.zeros((n_small, n_small))
        for i in range(n_small):
            for j in range(n_small):
                if i != j:
                    dist_small[i, j] = haversine_km(
                        small_lons[i], small_lats[i],
                        small_lons[j], small_lats[j]
                    )
                else:
                    dist_small[i, j] = np.inf
        
        small_merge_to_big = []
        small_need_cluster = []
        
        for i in range(n_small):
            d_big_min = dist_to_big[i, :].min()
            idx_big_nearest = dist_to_big[i, :].argmin()
            d_small_min = dist_small[i, :].min()
            
            if (d_big_min <= max_merge_to_big_km and 
                d_big_min < d_small_min * distance_ratio_threshold):
                small_merge_to_big.append((i, idx_big_nearest, d_big_min))
            else:
                small_need_cluster.append(i)
        
        merged_samples = 0
        for small_idx, big_idx, dist_km in small_merge_to_big:
            small_blk = small_blocks.iloc[small_idx][BLOCK_COL]
            big_blk = big_blocks.iloc[big_idx][BLOCK_COL]
            target_cluster = block_to_cluster[big_blk]
            block_to_cluster[small_blk] = target_cluster
            merged_samples += small_blocks.iloc[small_idx]["blk_count"]
        
        print(f"[小样本融入] {len(small_merge_to_big)}个板块 | {merged_samples}样本")
        
        # 剩余小样本聚类
        n_cluster_needed = len(small_need_cluster)
        if n_cluster_needed > 0:
            cluster_samples = sum(small_blocks.iloc[i]["blk_count"] for i in small_need_cluster)
            
            if cluster_samples < min_samples_per_small_cluster:
                for small_idx in small_need_cluster:
                    small_blk = small_blocks.iloc[small_idx][BLOCK_COL]
                    d_to_big = dist_to_big[small_idx, :]
                    nearest_big_idx = d_to_big.argmin()
                    big_blk = big_blocks.iloc[nearest_big_idx][BLOCK_COL]
                    target_cluster = block_to_cluster[big_blk]
                    block_to_cluster[small_blk] = target_cluster
            else:
                cluster_lons = small_blocks.iloc[small_need_cluster]["lon_med"].values
                cluster_lats = small_blocks.iloc[small_need_cluster]["lat_med"].values
                cluster_xy = lonlat_to_mercator(cluster_lons, cluster_lats)
                
                k_small = choose_k_small(
                    cluster_samples, target_small_samples_per_cluster,
                    k_small_min, k_small_max, n_cluster_needed,
                    min_samples_per_small_cluster
                )
                
                if k_small > 0 and k_small < n_cluster_needed:
                    from sklearn.cluster import KMeans
                    km = KMeans(n_clusters=k_small, random_state=random_state, n_init=10)
                    small_labels = km.fit_predict(cluster_xy)
                    
                    for local_idx, label in enumerate(small_labels):
                        small_idx = small_need_cluster[local_idx]
                        small_blk = small_blocks.iloc[small_idx][BLOCK_COL]
                        cluster_id = next_cluster_id + label
                        block_to_cluster[small_blk] = cluster_id
                        if cluster_id not in cluster_centers:
                            mask = small_labels == label
                            cluster_centers[cluster_id] = (
                                cluster_lons[mask].mean(),
                                cluster_lats[mask].mean()
                            )
                    
                    next_cluster_id += k_small
                    print(f"[KMeans聚类] k={k_small}")
                else:
                    for small_idx in small_need_cluster:
                        small_blk = small_blocks.iloc[small_idx][BLOCK_COL]
                        block_to_cluster[small_blk] = next_cluster_id
                        cluster_centers[next_cluster_id] = (
                            small_blocks.iloc[small_idx]["lon_med"],
                            small_blocks.iloc[small_idx]["lat_med"]
                        )
                        next_cluster_id += 1
    
    elif n_small > 0 and n_big == 0:
        total_small_samples = int(small_blocks["blk_count"].sum())
        if total_small_samples < min_samples_per_small_cluster:
            for idx, row in small_blocks.iterrows():
                block_to_cluster[row[BLOCK_COL]] = next_cluster_id
                cluster_centers[next_cluster_id] = (row["lon_med"], row["lat_med"])
                next_cluster_id += 1
        else:
            cluster_xy = lonlat_to_mercator(small_blocks["lon_med"].values, small_blocks["lat_med"].values)
            k_small = choose_k_small(
                total_small_samples, target_small_samples_per_cluster,
                k_small_min, k_small_max, n_small, min_samples_per_small_cluster
            )
            if k_small > 0 and k_small < n_small:
                from sklearn.cluster import KMeans
                km = KMeans(n_clusters=k_small, random_state=random_state, n_init=10)
                labels = km.fit_predict(cluster_xy)
                for idx, (_, row) in enumerate(small_blocks.iterrows()):
                    block_to_cluster[row[BLOCK_COL]] = labels[idx]
                    if labels[idx] not in cluster_centers:
                        cluster_centers[labels[idx]] = (row["lon_med"], row["lat_med"])
                next_cluster_id = k_small
            else:
                for idx, row in small_blocks.iterrows():
                    block_to_cluster[row[BLOCK_COL]] = next_cluster_id
                    cluster_centers[next_cluster_id] = (row["lon_med"], row["lat_med"])
                    next_cluster_id += 1
    
    # 6. 检查并合并不足100样本的簇
    df.loc[train_mask, "cluster"] = df.loc[train_mask, BLOCK_COL].map(block_to_cluster).astype("Int64")
    
    cluster_counts = df.loc[train_mask & df["cluster"].notna(), "cluster"].value_counts()
    small_clusters = cluster_counts[cluster_counts < min_count_big_block].index.tolist()
    big_clusters = cluster_counts[cluster_counts >= min_count_big_block].index.tolist()
    
    if len(small_clusters) > 0 and len(big_clusters) > 0:
        print(f"\n[合并小簇] {len(small_clusters)}个簇<100样本，合并到最近大簇...")
        
        small_cluster_centers = {}
        for cid in small_clusters:
            mask = train_mask & (df["cluster"] == cid) & df[LON_COL].notna() & df[LAT_COL].notna()
            if mask.any():
                small_cluster_centers[cid] = (
                    df.loc[mask, LON_COL].median(),
                    df.loc[mask, LAT_COL].median()
                )
        
        big_cluster_centers = {}
        for cid in big_clusters:
            if cid in cluster_centers:
                big_cluster_centers[cid] = cluster_centers[cid]
            else:
                mask = train_mask & (df["cluster"] == cid) & df[LON_COL].notna() & df[LAT_COL].notna()
                if mask.any():
                    big_cluster_centers[cid] = (
                        df.loc[mask, LON_COL].median(),
                        df.loc[mask, LAT_COL].median()
                    )
        
        merge_map = {}
        for small_cid, (slon, slat) in small_cluster_centers.items():
            min_dist = np.inf
            target_big_cid = None
            
            for big_cid, (blon, blat) in big_cluster_centers.items():
                dist = haversine_km(slon, slat, blon, blat)
                if dist < min_dist:
                    min_dist = dist
                    target_big_cid = big_cid
            
            if target_big_cid is not None:
                merge_map[small_cid] = target_big_cid
        
        for block, cid in list(block_to_cluster.items()):
            if cid in merge_map:
                block_to_cluster[block] = merge_map[cid]
        
        df.loc[train_mask, "cluster"] = df.loc[train_mask, BLOCK_COL].map(block_to_cluster).astype("Int64")
        
        final_cluster_counts = df.loc[train_mask & df["cluster"].notna(), "cluster"].value_counts()
        print(f"✅ 合并后: {len(final_cluster_counts)}个簇 | 最小{final_cluster_counts.min()}样本")
    
    # 7. 训练集无板块样本分配
    train_has_coord = train_mask & df[LON_COL].notna() & df[LAT_COL].notna()
    train_need_fallback = train_mask & df["cluster"].isna() & train_has_coord
    
    if train_need_fallback.any():
        final_cluster_ids = df.loc[train_mask & df["cluster"].notna(), "cluster"].unique()
        final_centers = {}
        for cid in final_cluster_ids:
            mask = train_mask & (df["cluster"] == cid) & df[LON_COL].notna() & df[LAT_COL].notna()
            if mask.any():
                final_centers[cid] = (
                    df.loc[mask, LON_COL].median(),
                    df.loc[mask, LAT_COL].median()
                )
        
        cids = sorted(final_centers.keys())
        center_lons = np.array([final_centers[c][0] for c in cids])
        center_lats = np.array([final_centers[c][1] for c in cids])
        
        fb_lons = df.loc[train_need_fallback, LON_COL].values
        fb_lats = df.loc[train_need_fallback, LAT_COL].values
        
        dists = np.vstack([
            haversine_km(fb_lons, fb_lats, clon, clat)
            for clon, clat in zip(center_lons, center_lats)
        ]).T
        nearest = np.argmin(dists, axis=1)
        df.loc[train_need_fallback, "cluster"] = [cids[i] for i in nearest]
    
    final_train_clusters = df.loc[train_mask & df["cluster"].notna(), "cluster"].nunique()
    
    # 8. 测试集最近邻分配
    print(f"\n[测试集分配] 找最近训练样本...")
    
    train_with_coord = train_mask & df[LON_COL].notna() & df[LAT_COL].notna()
    train_lons = df.loc[train_with_coord, LON_COL].values
    train_lats = df.loc[train_with_coord, LAT_COL].values
    train_clusters = df.loc[train_with_coord, "cluster"].values
    
    test_with_coord = test_mask & df[LON_COL].notna() & df[LAT_COL].notna()
    n_test_coord = test_with_coord.sum()
    
    if n_test_coord > 0:
        test_lons = df.loc[test_with_coord, LON_COL].values
        test_lats = df.loc[test_with_coord, LAT_COL].values
        test_indices = df[test_with_coord].index.values
        
        batch_size = 1000
        n_test_batches = int(np.ceil(n_test_coord / batch_size))
        
        for batch_idx in range(n_test_batches):
            start = batch_idx * batch_size
            end = min((batch_idx + 1) * batch_size, n_test_coord)
            
            batch_lons = test_lons[start:end]
            batch_lats = test_lats[start:end]
            batch_indices = test_indices[start:end]
            
            dists = np.zeros((len(batch_lons), len(train_lons)))
            for i, (tlon, tlat) in enumerate(zip(batch_lons, batch_lats)):
                dists[i, :] = haversine_km(tlon, tlat, train_lons, train_lats)
            
            nearest_train_idx = np.argmin(dists, axis=1)
            
            for i, test_idx in enumerate(batch_indices):
                nearest_cluster = train_clusters[nearest_train_idx[i]]
                df.loc[test_idx, "cluster"] = nearest_cluster
        
        print(f"✅ 测试集{n_test_coord}样本已分配")
    
    # 9. 缺失标记
    df["cluster_missing"] = df["cluster"].isna().astype(int)
    
    # 10. One-Hot（删基准）
    df, baseline, ohe_cols = one_hot_drop_baseline(df, "cluster", "cluster")
    
    # 11. 删除原始列
    drop_cols = [BLOCK_COL, "cluster"]
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors="ignore")
    
    # 12. 清理列名：删除面积列的²符号
    df.columns = df.columns.str.replace('m²', '', regex=False)
    
    # 13. 输出
    p_in = Path(path_in)
    path_out = str(p_in.with_name(p_in.stem + out_suffix))
    df.to_csv(path_out, index=False, encoding=enc or "utf-8-sig")
    
    print(f"\n{'='*60}")
    print(f"✅ 完成！最终簇数: {final_train_clusters} | One-Hot维度: {len(ohe_cols)}")
    print(f"📁 输出: {Path(path_out).name}")
    print(f"{'='*60}")

if __name__ == "__main__":
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        main()

训练集聚类 + 测试集最近邻分配（每簇≥100样本）

[清理面积列] 找到 2 个面积列: ['面积', '面积.1']
  面积 已是数值类型，跳过
  处理: 面积.1
    ✅ 转换完成

总样本: 97709 | 训练: 78184 | 测试: 19525

[训练集板块] 大样本: 212个 | 小样本: 692个
[小样本融入] 164个板块 | 6376样本
[KMeans聚类] k=88

[合并小簇] 44个簇<100样本，合并到最近大簇...
✅ 合并后: 256个簇 | 最小100样本

[测试集分配] 找最近训练样本...
✅ 测试集19525样本已分配

✅ 完成！最终簇数: 256 | One-Hot维度: 255
📁 输出: ruc_Class25Q2_train_rent91.csv


In [1]:
# -*- coding: utf-8 -*-
"""
租房价格预测 - 对数月租线性回归建模
改进：
1. 计算Price = 月租 × 付款方式（仅用于展示）
2. 对月租取对数（处理右偏）
3. 根据is_train列划分数据集，只从训练集删除异常值
4. 使用月租作为目标变量
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import joblib

# ===================== 配置区 =====================
path_in = r"H:\HW\ruc_Class25Q2_train_rent91.csv"

random_state = 42
cv_folds = 6

# 明确指定连续变量（根据提供的列名）
CONTINUOUS_FEATURES = [
    '面积','容积率',
    '室',  
    '房屋总数', 
    '楼栋总数',
    '绿化率',
    '物业费'
]

# 特征工程开关
enable_log_transform = True      
enable_interaction = False        
enable_binning = True            

# 🔥 删除不需要的特征
FEATURES_TO_REMOVE = [
    
]

ENGINEERED_FEATURES_TO_REMOVE = [
   '容积率',           # 找到实际列名替换
    '室',              # 或 '卧室数', '房间数' 等
    '房屋总数',         # 找到实际列名替换
    '楼栋总数',         # 找到实际列名替换
    '绿化率',           # 找到实际列名替换
    '物业费',   
]

# =================================================

print("=" * 80)
print("租房价格预测 - 对数月租线性回归建模")
print("=" * 80)
print("🎯 改进：根据is_train划分 + 对数变换 + 训练集异常值清理")
print("=" * 80)


# ============ 1. 数据加载 + 数据清洗 ============
print("\n" + "=" * 80)
print("Step 1: 数据加载 + 数据清洗")
print("=" * 80)

df = pd.read_csv(path_in, encoding='utf-8')
print(f"原始数据形状: {df.shape}")

# 检查is_train列
if 'is_train' not in df.columns:
    raise ValueError("数据中没有'is_train'列！")

# 🔥 1.0 清洗面积列（去掉m²等单位）
print("\n" + "-" * 80)
print("1.0 清洗面积列（去除单位）")
print("-" * 80)

area_cols = [col for col in df.columns if '面积' in col]
if area_cols:
    print(f"找到 {len(area_cols)} 个面积列: {area_cols}")
    for col in area_cols:
        if df[col].dtype == 'object':
            print(f"  处理: {col}")
            df[col] = df[col].astype(str).str.replace(r'[m²㎡平米方]+', '', regex=True)
            df[col] = df[col].str.replace('平方米', '', regex=False)
            df[col] = df[col].str.replace('平方', '', regex=False)
            df[col] = df[col].str.strip()
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
            missing_count = df[col].isna().sum()
            if missing_count > 0:
                median_val = df[col].median()
                df[col].fillna(median_val, inplace=True)
                print(f"    ✅ 转换完成，填充了 {missing_count} 个缺失值（中位数={median_val:.2f}）")
            else:
                print(f"    ✅ 转换完成")
        else:
            print(f"  {col} 已是数值类型，跳过")

if '月租' not in df.columns:
    raise ValueError("数据中没有'月租'列！")

# 删除城市特征
city_cols = [col for col in df.columns if col.startswith('city_') or col == '城市']
if city_cols:
    print(f"\n删除 {len(city_cols)} 个城市特征: {city_cols}")
    df = df.drop(columns=city_cols)

# 删除不需要的特征
cols_to_remove = [col for col in FEATURES_TO_REMOVE if col in df.columns]
if cols_to_remove:
    print(f"删除特征: {cols_to_remove}")
    df = df.drop(columns=cols_to_remove)

# 验证连续特征
missing_features = [f for f in CONTINUOUS_FEATURES if f not in df.columns]
if missing_features:
    print(f"\n⚠️ 以下特征不存在: {missing_features}")
    CONTINUOUS_FEATURES = [f for f in CONTINUOUS_FEATURES if f in df.columns]

print(f"\n确认的连续特征({len(CONTINUOUS_FEATURES)}个): {CONTINUOUS_FEATURES}")

# 🔥 1.1 计算Price（仅用于展示分析）
print("\n" + "-" * 80)
print("1.1 计算Price（月租 × 付款方式，仅用于展示）")
print("-" * 80)

payment_months = {
    '付款方式_月付价': 1,
    '付款方式_双月付价': 2,
    '付款方式_半年付价': 6,
    '付款方式_年付价': 12
}

df['付款月份'] = 3  # 默认季付（基准类别）
for col, months in payment_months.items():
    if col in df.columns:
        df.loc[df[col] == 1, '付款月份'] = months

df['Price'] = df['月租'] * df['付款月份']

print(f"Price计算完成（仅用于分析）:")
print(f"  月租范围: [{df['月租'].min():.0f}, {df['月租'].max():.0f}]")
print(f"  Price范围: [{df['Price'].min():.0f}, {df['Price'].max():.0f}]")

# 🔥 1.2 对月租取对数（目标变量）
print("\n" + "-" * 80)
print("1.2 对月租取对数（处理右偏分布）")
print("-" * 80)

print(f"原始月租统计:")
print(f"  均值: {df['月租'].mean():.2f}")
print(f"  标准差: {df['月租'].std():.2f}")
print(f"  范围: [{df['月租'].min():.2f}, {df['月租'].max():.2f}]")
print(f"  偏度: {df['月租'].skew():.2f} (>1为右偏)")

# 取对数
df['月租_log'] = np.log1p(df['月租'])

print(f"\n对数月租统计:")
print(f"  均值: {df['月租_log'].mean():.4f}")
print(f"  标准差: {df['月租_log'].std():.4f}")
print(f"  范围: [{df['月租_log'].min():.4f}, {df['月租_log'].max():.4f}]")
print(f"  偏度: {df['月租_log'].skew():.2f} (对数后接近正态)")

# 🔥 1.3 检测对数月租的异常值（仅标记，不删除）
print("\n" + "-" * 80)
print("1.3 检测对数月租的异常值（3σ原则 - 基于训练集）")
print("-" * 80)

# 🔥 只基于训练集计算异常值边界
train_mask = df['is_train'] == 1
mean_log_train = df.loc[train_mask, '月租_log'].mean()
std_log_train = df.loc[train_mask, '月租_log'].std()
lower_bound = mean_log_train - 3 * std_log_train
upper_bound = mean_log_train + 3 * std_log_train

print(f"训练集对数月租统计:")
print(f"  均值: {mean_log_train:.4f}")
print(f"  标准差: {std_log_train:.4f}")
print(f"  3σ范围: [{lower_bound:.4f}, {upper_bound:.4f}]")
print(f"  对应原始月租: [{np.expm1(lower_bound):.0f}, {np.expm1(upper_bound):.0f}]")

# 🔥 只标记异常值，不删除
df['is_outlier'] = ((df['月租_log'] < lower_bound) | (df['月租_log'] > upper_bound)).astype(int)

n_outliers_total = df['is_outlier'].sum()
n_outliers_train = df.loc[train_mask, 'is_outlier'].sum()
n_outliers_test = df.loc[~train_mask, 'is_outlier'].sum()

print(f"\n异常值统计:")
print(f"  总异常样本数: {n_outliers_total} ({n_outliers_total/len(df)*100:.2f}%)")
print(f"  训练集异常值: {n_outliers_train} ({n_outliers_train/train_mask.sum()*100:.2f}%)")
print(f"  测试集异常值: {n_outliers_test} ({n_outliers_test/(~train_mask).sum()*100:.2f}%)")

if n_outliers_total > 0:
    outlier_mask = df['is_outlier'] == 1
    print(f"  异常月租范围: [{df[outlier_mask]['月租'].min():.0f}, {df[outlier_mask]['月租'].max():.0f}]")
    print(f"  ⚠️ 将仅从训练集移除异常值，测试集保留所有样本")
else:
    print(f"  ✅ 无异常值")

# 🔥 分离特征和目标（使用对数月租作为目标）
y_log = df['月租_log'].copy()
y_original = df['月租'].copy()

# 🔥 删除月租、Price、付款月份、is_train、is_outlier等不应作为特征的列
cols_to_exclude = ['月租', '月租_log', 'Price', '付款月份', 'is_train', 'is_outlier']
X = df.drop(columns=[col for col in cols_to_exclude if col in df.columns])

print(f"\n✅ 数据准备完成:")
print(f"  样本数: {len(df)}")
print(f"  训练集: {train_mask.sum()} ({train_mask.sum()/len(df)*100:.1f}%)")
print(f"  测试集: {(~train_mask).sum()} ({(~train_mask).sum()/len(df)*100:.1f}%)")
print(f"  目标变量: 月租_log (对数月租)")
print(f"  特征数: {X.shape[1]}")


# ============ 2. 识别特征类型并清洗连续特征 ============
print("\n" + "=" * 80)
print("Step 2: 特征类型识别与连续特征清洗")
print("=" * 80)

continuous_features = [f for f in CONTINUOUS_FEATURES if f in X.columns]

onehot_features = [col for col in X.columns 
                   if col not in continuous_features]

print(f"连续特征: {len(continuous_features)} 个")
print(f"  {continuous_features}")
print(f"One-Hot/二值特征: {len(onehot_features)} 个")

# 🔥 2.1 清洗所有连续特征（确保数值类型）
print("\n" + "-" * 80)
print("2.1 清洗连续特征（转换为数值类型）")
print("-" * 80)

for col in continuous_features:
    if X[col].dtype == 'object':
        print(f"⚠️ {col} 是字符串类型，正在转换...")
        
        # 🔥 对面积列特殊处理（去除m²、㎡等单位）
        if '面积' in col:
            X[col] = X[col].astype(str).str.replace(r'[m²㎡平米方]+', '', regex=True)
            X[col] = X[col].str.replace('平方米', '', regex=False)
            X[col] = X[col].str.replace('平方', '', regex=False)
        
        # 去除其他常见单位符号
        X[col] = X[col].astype(str).str.replace('%', '', regex=False)
        X[col] = X[col].str.replace('元', '', regex=False)
        X[col] = X[col].str.replace(',', '', regex=False)
        X[col] = X[col].str.strip()
        
        # 转换为数值
        X[col] = pd.to_numeric(X[col], errors='coerce')
        
        # 填充缺失值
        missing_count = X[col].isna().sum()
        if missing_count > 0:
            median_val = X[col].median()
            X[col].fillna(median_val, inplace=True)
            print(f"   {col}: 转换完成，填充了 {missing_count} 个缺失值（用中位数{median_val:.2f}）")
        else:
            print(f"   {col}: 转换完成")
    else:
        # 检查是否有缺失值
        missing_count = X[col].isna().sum()
        if missing_count > 0:
            median_val = X[col].median()
            X[col].fillna(median_val, inplace=True)
            print(f"   {col}: 填充了 {missing_count} 个缺失值（用中位数{median_val:.2f}）")

print("\n✅ 所有连续特征已转换为数值类型")


# ============ 3. 特征工程 ============
print("\n" + "=" * 80)
print("Step 3: 特征工程")
print("=" * 80)

X_engineered = X.copy()
new_continuous_features = []

# 3.1 对数转换
if enable_log_transform:
    print("\n3.1 对数转换")
    for col in continuous_features:
        if (X_engineered[col] > 0).all():
            skewness = X_engineered[col].skew()
            if skewness > 2:
                new_col = f'{col}_log'
                X_engineered[new_col] = np.log1p(X_engineered[col])
                new_continuous_features.append(new_col)
                print(f"  {col} (偏度={skewness:.2f}) → {new_col}")
        else:
            print(f"  ⚠️ {col} 包含非正值，跳过对数转换")

# 3.2 交互项
if enable_interaction:
    print("\n3.2 交互项")
    
    # 面积与室数的交互
    if '面积' in continuous_features and '室' in continuous_features:
        X_engineered['室_adjusted'] = X_engineered['室'].replace(0, 1)
        new_col = '面积_per_室'
        X_engineered[new_col] = X_engineered['面积.1'] / X_engineered['室_adjusted']
        new_continuous_features.append(new_col)
        print(f"  ✅ {new_col} = 面积/室")
# 3.3 分箱
if enable_binning:
    area_col = None
    for col in continuous_features:
        if '面积' in col:
            area_col = col
            break
    
    if area_col:
        print("\n3.3 分箱")
        X_engineered['面积_小户型'] = (X_engineered[area_col] < 50).astype(int)
        X_engineered['面积_中户型'] = ((X_engineered[area_col] >= 50) & (X_engineered[area_col] <= 80)).astype(int)
        X_engineered['面积_大户型'] = (X_engineered[area_col] > 80).astype(int)
        onehot_features.extend(['面积_小户型', '面积_中户型', '面积_大户型'])
        print(f"  面积分箱完成 (<50, 50-80, >80)")

all_continuous = continuous_features + new_continuous_features

print(f"\n✅ 特征工程汇总:")
print(f"  原始连续特征: {len(continuous_features)} 个")
print(f"  新增连续特征: {len(new_continuous_features)} 个")
print(f"  连续特征总计: {len(all_continuous)} 个")


# ============ 3.5 删除共线特征 ============
print("\n" + "=" * 80)
print("Step 3.5: 删除共线特征（可选）")
print("=" * 80)

if ENGINEERED_FEATURES_TO_REMOVE:
    features_to_remove_actual = [f for f in ENGINEERED_FEATURES_TO_REMOVE if f in X_engineered.columns]
    if features_to_remove_actual:
        print(f"删除 {len(features_to_remove_actual)} 个共线特征: {features_to_remove_actual}")
        X_engineered = X_engineered.drop(columns=features_to_remove_actual)
        all_continuous = [f for f in all_continuous if f not in features_to_remove_actual]
        print(f"✅ 剩余连续特征: {len(all_continuous)} 个")
else:
    print("暂无预设的共线特征需要删除")


# ============ 4. 根据is_train划分数据集（只在训练集删除异常值）============
print("\n" + "=" * 80)
print("Step 4: 根据is_train划分数据集（训练集移除异常值）")
print("=" * 80)

# 🔥 根据is_train列划分
train_mask = df['is_train'] == 1
test_mask = df['is_train'] == 0

# 原始划分
X_train_raw = X_engineered[train_mask].copy()
y_train_log_raw = y_log[train_mask].copy()
y_train_orig_raw = y_original[train_mask].copy()
outlier_train = df.loc[train_mask, 'is_outlier'].values

X_test = X_engineered[test_mask].copy()
y_test_log = y_log[test_mask].copy()
y_test_orig = y_original[test_mask].copy()
outlier_test = df.loc[test_mask, 'is_outlier'].values

print(f"原始划分:")
print(f"  训练集: {len(X_train_raw)} 样本 | 测试集: {len(X_test)} 样本")

# 🔥 从训练集中移除异常值
train_non_outlier_mask = outlier_train == 0
X_train = X_train_raw[train_non_outlier_mask].copy()
y_train_log = y_train_log_raw[train_non_outlier_mask].copy()
y_train_orig = y_train_orig_raw[train_non_outlier_mask].copy()

n_outliers_train_removed = outlier_train.sum()
n_outliers_test_kept = outlier_test.sum()

print(f"\n异常值处理:")
print(f"  训练集异常值: {n_outliers_train_removed} ({n_outliers_train_removed/len(X_train_raw)*100:.2f}%) - 已删除 ❌")
print(f"  测试集异常值: {n_outliers_test_kept} ({n_outliers_test_kept/len(X_test)*100:.2f}%) - 保留 ✅")

print(f"\n✅ 最终数据集:")
print(f"  训练集: {len(X_train)} 样本 ({len(X_train)/(len(X_train)+len(X_test))*100:.1f}%)")
print(f"  测试集: {len(X_test)} 样本 ({len(X_test)/(len(X_train)+len(X_test))*100:.1f}%)")


# ============ 5. 标准化 ============
print("\n" + "=" * 80)
print("Step 5: 标准化")
print("=" * 80)

all_continuous_exist = [col for col in all_continuous if col in X_train.columns]
print(f"标准化特征数: {len(all_continuous_exist)} 个")

scaler = StandardScaler()
X_train[all_continuous_exist] = scaler.fit_transform(X_train[all_continuous_exist])
X_test[all_continuous_exist] = scaler.transform(X_test[all_continuous_exist])

print("✅ 标准化完成")


# ============ 5.5 VIF验证 ============
print("\n" + "=" * 80)
print("Step 5.5: VIF验证")
print("=" * 80)

if len(all_continuous_exist) <= 30:
    print(f"计算 {len(all_continuous_exist)} 个连续特征的VIF...\n")
    
    try:
        X_vif = X_train[all_continuous_exist].values
        vif_data = pd.DataFrame()
        vif_data["Feature"] = all_continuous_exist
        vif_data["VIF"] = [variance_inflation_factor(X_vif, i) for i in range(len(all_continuous_exist))]
        vif_data = vif_data.sort_values('VIF', ascending=False)
        
        print("VIF检测结果:")
        print("=" * 60)
        print(vif_data.to_string(index=False))
        print("=" * 60)
        
        vif_excellent = len(vif_data[vif_data['VIF'] < 2])
        vif_good = len(vif_data[(vif_data['VIF'] >= 2) & (vif_data['VIF'] < 5)])
        vif_acceptable = len(vif_data[(vif_data['VIF'] >= 5) & (vif_data['VIF'] < 10)])
        vif_high = len(vif_data[vif_data['VIF'] >= 10])
        
        print(f"\n📊 VIF质量:")
        print(f"  VIF < 2: {vif_excellent} 个 ({vif_excellent/len(vif_data)*100:.1f}%)")
        print(f"  2 ≤ VIF < 5: {vif_good} 个 ({vif_good/len(vif_data)*100:.1f}%)")
        print(f"  5 ≤ VIF < 10: {vif_acceptable} 个 ({vif_acceptable/len(vif_data)*100:.1f}%)")
        print(f"  VIF ≥ 10: {vif_high} 个 ({vif_high/len(vif_data)*100:.1f}%)")
        
        if vif_high == 0 and vif_acceptable == 0:
            print(f"\n🎉 完美！所有VIF < 5")
        elif vif_high > 0:
            print(f"\n⚠️ 建议删除VIF≥10的特征以改善共线性")
        
        vif_data.to_csv('rent_vif_log_月租.csv', index=False, encoding='utf-8-sig')
        
    except Exception as e:
        print(f"\n⚠️ VIF计算失败: {e}")
else:
    print(f"连续特征过多({len(all_continuous_exist)}个)，跳过VIF计算")


# ============ 6. 特征选择 ============
print("\n" + "=" * 80)
print("Step 6: 特征选择")
print("=" * 80)

selected_continuous = all_continuous_exist.copy()

correlation_threshold = 0.01
correlations = []
for col in selected_continuous:
    corr = np.corrcoef(X_train[col], y_train_log)[0, 1]
    correlations.append({'feature': col, 'correlation': corr})

corr_df = pd.DataFrame(correlations)
corr_df['abs_corr'] = corr_df['correlation'].abs()

print(f"\nTop 10 高相关特征:")
top_corr = corr_df.nlargest(min(10, len(corr_df)), 'abs_corr')
for idx, row in top_corr.iterrows():
    direction = "📈" if row['correlation'] > 0 else "📉"
    print(f"  {direction} {row['feature']}: {row['correlation']:.4f}")

low_corr = corr_df[corr_df['abs_corr'] < correlation_threshold]
if len(low_corr) > 0:
    print(f"\n低相关性特征: {len(low_corr)} 个，已删除")
    print(f"  删除: {low_corr['feature'].tolist()}")
    selected_continuous = [f for f in selected_continuous if f not in low_corr['feature'].tolist()]

selected_features = selected_continuous + onehot_features

X_train_final = X_train[selected_features]
X_test_final = X_test[selected_features]

print(f"\n✅ 最终特征: {len(selected_features)} 个")
print(f"  连续: {len(selected_continuous)} | One-Hot/二值: {len(onehot_features)}")


# ============ 7. 模型训练（对数月租）============
print("\n" + "=" * 80)
print("Step 7: 模型训练（对数月租建模）")
print("=" * 80)

def evaluate_model_log(model, X_train, X_test, y_train_log, y_test_log, y_test_orig, model_name):
    """评估对数月租模型（统一在原始空间计算所有指标）"""
    
    # ============ 1. 训练集预测 ============
    y_train_pred_log = model.predict(X_train)
    y_train_pred = np.expm1(y_train_pred_log)  # 转回原始尺度
    y_train_actual = np.expm1(y_train_log)
    
    mae_train = mean_absolute_error(y_train_actual, y_train_pred)
    rmse_train = np.sqrt(mean_squared_error(y_train_actual, y_train_pred))
    r2_train = r2_score(y_train_actual, y_train_pred)
    
    # ============ 2. 测试集预测 ============
    y_test_pred_log = model.predict(X_test)
    y_test_pred = np.expm1(y_test_pred_log)  # 转回原始尺度
    
    mae_test = mean_absolute_error(y_test_orig, y_test_pred)
    rmse_test = np.sqrt(mean_squared_error(y_test_orig, y_test_pred))
    r2_test = r2_score(y_test_orig, y_test_pred)
    
    # ============ 3. 交叉验证（在原始空间计算MAE和R²）============
    kfold = KFold(n_splits=cv_folds, shuffle=True, random_state=random_state)
    cv_scores_mae = []
    cv_scores_r2 = []
    
    for train_idx, val_idx in kfold.split(X_train):
        X_cv_train, X_cv_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_cv_train_log, y_cv_val_log = y_train_log.iloc[train_idx], y_train_log.iloc[val_idx]
        
        # 训练模型
        model.fit(X_cv_train, y_cv_train_log)
        y_cv_pred_log = model.predict(X_cv_val)
        
        # 🔥 转回原始尺度计算指标
        y_cv_pred = np.expm1(y_cv_pred_log)
        y_cv_actual = np.expm1(y_cv_val_log)
        
        cv_scores_mae.append(mean_absolute_error(y_cv_actual, y_cv_pred))
        cv_scores_r2.append(r2_score(y_cv_actual, y_cv_pred))
    
    mae_cv = np.mean(cv_scores_mae)
    mae_cv_std = np.std(cv_scores_mae)
    r2_cv = np.mean(cv_scores_r2)
    r2_cv_std = np.std(cv_scores_r2)
    
    # ============ 4. 返回完整结果 ============
    return {
        'Model': model_name,
        'MAE_train': mae_train,
        'RMSE_train': rmse_train,
        'R2_train': r2_train,
        'MAE_test': mae_test,
        'RMSE_test': rmse_test,
        'R2_test': r2_test,
        'MAE_cv': mae_cv,
        'MAE_cv_std': mae_cv_std,
        'R2_cv': r2_cv,
        'R2_cv_std': r2_cv_std,
        'Overfit': mae_test - mae_train
    }

results = []  

# 7.1 OLS (对数月租)
print("\n7.1 OLS（对数月租）")
ols_model = LinearRegression()
ols_model.fit(X_train_final, y_train_log)
ols_result = evaluate_model_log(ols_model, X_train_final, X_test_final, 
                                 y_train_log, y_test_log, y_test_orig, 'OLS_对数月租')
results.append(ols_result)
print(f"  测试集: MAE={ols_result['MAE_test']:.2f} | RMSE={ols_result['RMSE_test']:.2f} | R²={ols_result['R2_test']:.4f}")
print(f"  交叉验证: MAE={ols_result['MAE_cv']:.2f}±{ols_result['MAE_cv_std']:.2f} | R²={ols_result['R2_cv']:.4f}")

# 7.2 Ridge (对数月租)
print("\n7.2 Ridge（对数月租）")
ridge_params = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]}
ridge_grid = GridSearchCV(Ridge(random_state=random_state), ridge_params, 
                          cv=cv_folds, scoring='neg_mean_absolute_error', n_jobs=-1)
ridge_grid.fit(X_train_final, y_train_log)
ridge_model = ridge_grid.best_estimator_
ridge_result = evaluate_model_log(ridge_model, X_train_final, X_test_final, 
                                   y_train_log, y_test_log, y_test_orig, 'Ridge_对数月租')
results.append(ridge_result)
print(f"  最佳alpha: {ridge_grid.best_params_['alpha']}")
print(f"  测试集: MAE={ridge_result['MAE_test']:.2f} | RMSE={ridge_result['RMSE_test']:.2f} | R²={ridge_result['R2_test']:.4f}")
print(f"  交叉验证: MAE={ridge_result['MAE_cv']:.2f}±{ridge_result['MAE_cv_std']:.2f} | R²={ridge_result['R2_cv']:.4f}")

# 7.3 Lasso (对数月租)
print("\n7.3 Lasso（对数月租）")
lasso_params = {'alpha': [ 0.001, 0.01, 0.1]}
lasso_grid = GridSearchCV(
    Lasso(random_state=random_state, max_iter=5000), 
    lasso_params, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
)
lasso_grid.fit(X_train_final, y_train_log)
lasso_model = lasso_grid.best_estimator_
lasso_result = evaluate_model_log(lasso_model, X_train_final, X_test_final, 
                                   y_train_log, y_test_log, y_test_orig, 'Lasso_对数月租')
results.append(lasso_result)

n_nonzero = np.sum(lasso_model.coef_ != 0)
print(f"  最佳alpha: {lasso_grid.best_params_['alpha']}")
print(f"  非零系数: {n_nonzero}/{len(lasso_model.coef_)} ({n_nonzero/len(lasso_model.coef_)*100:.1f}%)")
print(f"  测试集: MAE={lasso_result['MAE_test']:.2f} | RMSE={lasso_result['RMSE_test']:.2f} | R²={lasso_result['R2_test']:.4f}")
print(f"  交叉验证: MAE={lasso_result['MAE_cv']:.2f}±{lasso_result['MAE_cv_std']:.2f} | R²={lasso_result['R2_cv']:.4f}")

# 7.4 ElasticNet (对数月租)
print("\n7.4 ElasticNet（对数月租）")
elastic_params = {'alpha': [0.001, 0.01, 0.1], 'l1_ratio': [0.3, 0.5, 0.7]}
elastic_grid = GridSearchCV(
    ElasticNet(random_state=random_state, max_iter=3000),
    elastic_params, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
)
elastic_grid.fit(X_train_final, y_train_log)
elastic_model = elastic_grid.best_estimator_
elastic_result = evaluate_model_log(elastic_model, X_train_final, X_test_final, 
                                     y_train_log, y_test_log, y_test_orig, 'ElasticNet_对数月租')
results.append(elastic_result)
print(f"  最佳参数: alpha={elastic_grid.best_params_['alpha']}, l1_ratio={elastic_grid.best_params_['l1_ratio']}")
print(f"  测试集: MAE={elastic_result['MAE_test']:.2f} | RMSE={elastic_result['RMSE_test']:.2f} | R²={elastic_result['R2_test']:.4f}")
print(f"  交叉验证: MAE={elastic_result['MAE_cv']:.2f}±{elastic_result['MAE_cv_std']:.2f} | R²={elastic_result['R2_cv']:.4f}")


# ============ 8. 结果汇总 ============
print("\n" + "=" * 80)
print("Step 8: 结果汇总")
print("=" * 80)

results_df = pd.DataFrame(results)

# 找出最佳模型
best_idx = results_df['MAE_test'].idxmin()
best_model_name = results_df.loc[best_idx, 'Model']
best_result = results_df.loc[best_idx]

print(f"\n{'🏆' * 40}")
print(f"最佳模型: {best_model_name}")
print(f"{'🏆' * 40}")

# ========== 核心性能指标表格 ==========
print("\n" + "=" * 80)
print("📊 核心性能指标（最佳模型）")
print("=" * 80)

performance_table = pd.DataFrame({
    '评估维度': ['样本内性能', '样本外性能', '6折交叉验证'],
    'MAE': [
        f"{best_result['MAE_train']:,.0f}",
        f"{best_result['MAE_test']:,.0f}",
        f"{best_result['MAE_cv']:,.0f} ± {best_result['MAE_cv_std']:,.0f}"
    ],
    'RMSE': [
        f"{best_result['RMSE_train']:,.0f}",
        f"{best_result['RMSE_test']:,.0f}",
        "N/A"
    ],
    'R²': [
        f"{best_result['R2_train']:.4f}",
        f"{best_result['R2_test']:.4f}",
        f"{best_result['R2_cv']:.4f}"
    ]
})

print(performance_table.to_string(index=False))

# 详细指标
print(f"\n" + "=" * 80)
print("📈 详细指标")
print("=" * 80)
print(f"【样本内性能（训练集）】")
print(f"  MAE:  {best_result['MAE_train']:>15,.0f} 元")
print(f"  RMSE: {best_result['RMSE_train']:>15,.0f} 元")
print(f"  R²:   {best_result['R2_train']:>15.4f}")

print(f"\n【样本外性能（测试集）】")
print(f"  MAE:  {best_result['MAE_test']:>15,.0f} 元")
print(f"  RMSE: {best_result['RMSE_test']:>15,.0f} 元")
print(f"  R²:   {best_result['R2_test']:>15.4f}")
print(f"  相对误差: {best_result['MAE_test']/y_original.mean()*100:>11.1f}%")

print(f"\n【6折交叉验证】")
print(f"  MAE:  {best_result['MAE_cv']:>15,.0f} ± {best_result['MAE_cv_std']:,.0f} 元")
print(f"  R²:   {best_result['R2_cv']:>15.4f}")

print(f"\n【模型稳定性】")
overfit_amount = best_result['Overfit']
overfit_pct = (best_result['MAE_test'] / best_result['MAE_train'] - 1) * 100
print(f"  过拟合量: {overfit_amount:>14,.0f} 元 ({overfit_pct:+.1f}%)")

if abs(overfit_pct) < 5:
    overfit_level = "🎉 优秀（几乎无过拟合）"
elif abs(overfit_pct) < 10:
    overfit_level = "✅ 良好"
elif abs(overfit_pct) < 20:
    overfit_level = "⚠️ 可接受"
else:
    overfit_level = "❌ 过拟合严重"

print(f"  稳定性评价: {overfit_level}")

# 数据集信息
print(f"\n" + "=" * 80)
print("📦 数据集信息")
print("=" * 80)
print(f"  原始数据量: {len(df):>15,} 样本")
print(f"  训练集原始: {len(X_train_raw):>15,} 样本")
print(f"  训练集异常值: {n_outliers_train_removed:>13,} 样本 ({n_outliers_train_removed/len(X_train_raw)*100:.2f}%) - 已删除 ❌")
print(f"  训练集实际: {len(X_train):>15,} 样本 ✅")
print(f"  测试集: {len(X_test):>19,} 样本（含{n_outliers_test_kept}个异常值）✅")
print(f"  特征数量: {len(selected_features):>17,} 个")

# 性能对比表
print("\n" + "=" * 80)
print("📊 所有模型性能对比（对数月租建模）")
print("=" * 80)

comparison = pd.DataFrame({
    '模型': [r['Model'] for r in results],
    '样本内MAE': [f"{r['MAE_train']:,.0f}" for r in results],
    '样本外MAE': [f"{r['MAE_test']:,.0f}" for r in results],
    '样本外RMSE': [f"{r['RMSE_test']:,.0f}" for r in results],
    'CV_MAE': [f"{r['MAE_cv']:,.0f}±{r['MAE_cv_std']:.0f}" for r in results],
    '测试R²': [f"{r['R2_test']:.4f}" for r in results],
    '相对误差%': [f"{r['MAE_test']/y_original.mean()*100:.1f}" for r in results]
})

print(comparison.to_string(index=False))

# 保存结果
results_df.to_csv('rent_model_results_log_月租.csv', index=False, encoding='utf-8-sig')
comparison.to_csv('rent_model_comparison_log_月租.csv', index=False, encoding='utf-8-sig')

# 保存最佳模型
if 'OLS' in best_model_name:
    best_model = ols_model
elif 'Lasso' in best_model_name:
    best_model = lasso_model
elif 'Ridge' in best_model_name:
    best_model = ridge_model
else:
    best_model = elastic_model

joblib.dump(best_model, f'rent_best_model_log_月租.pkl')
joblib.dump(ols_model, 'rent_model_OLS_log_月租.pkl')
joblib.dump(ridge_model, 'rent_model_Ridge_log_月租.pkl')
joblib.dump(lasso_model, 'rent_model_Lasso_log_月租.pkl')
joblib.dump(elastic_model, 'rent_model_ElasticNet_log_月租.pkl')
joblib.dump(scaler, 'rent_scaler_log_月租.pkl')
joblib.dump(selected_features, 'rent_selected_features_log_月租.pkl')

print("\n" + "=" * 80)
print("💾 保存的文件")
print("=" * 80)
print("  📊 rent_model_results_log_月租.csv")
print("  📊 rent_model_comparison_log_月租.csv")
print("  📊 rent_vif_log_月租.csv")
print(f"  💾 rent_best_model_log_月租.pkl ({best_model_name})")
print("  💾 rent_model_OLS_log_月租.pkl")
print("  💾 rent_model_Ridge_log_月租.pkl")
print("  💾 rent_model_Lasso_log_月租.pkl")
print("  💾 rent_model_ElasticNet_log_月租.pkl")
print("  💾 rent_scaler_log_月租.pkl")
print("  💾 rent_selected_features_log_月租.pkl")


# ============ 9. 总结报告 ============
print("\n" + "=" * 80)
print("🎉 租房价格预测建模完成！")
print("=" * 80)

print(f"\n📋 数据处理总结:")
print(f"  原始数据量: {len(df):,} 样本")
print(f"  数据集划分: 根据 is_train 列自动识别")
print(f"  目标变量: 月租（对数变换）")
print(f"  Price计算: 月租 × 付款方式（仅用于展示分析）")
print(f"  对数变换: 月租偏度 → 月租_log偏度 {df['月租_log'].skew():.2f}")
print(f"  异常值清理: 基于训练集3σ原则")
print(f"    - 训练集删除: {n_outliers_train_removed:,} 样本 ({n_outliers_train_removed/len(X_train_raw)*100:.2f}%) ❌")
print(f"    - 测试集保留: {n_outliers_test_kept:,} 样本 ({n_outliers_test_kept/len(X_test)*100:.2f}%) ✅")
print(f"  ✅ 训练集清理后: {len(X_train):,} 样本")
print(f"  ✅ 测试集保留: {len(X_test):,} 样本")
print(f"  特征工程: {X.shape[1]} → {len(selected_features)} 个特征")

print(f"\n🏆 最佳模型: {best_model_name}")
print(f"  【样本内】  MAE = {best_result['MAE_train']:,.0f},  RMSE = {best_result['RMSE_train']:,.0f},  R² = {best_result['R2_train']:.4f}")
print(f"  【样本外】  MAE = {best_result['MAE_test']:,.0f},  RMSE = {best_result['RMSE_test']:,.0f},  R² = {best_result['R2_test']:.4f}")
print(f"  【交叉验证】MAE = {best_result['MAE_cv']:,.0f} ± {best_result['MAE_cv_std']:,.0f},  R² = {best_result['R2_cv']:.4f}")
print(f"  相对误差: {best_result['MAE_test']/y_original.mean()*100:.1f}%")

# 性能评价
relative_error_pct = best_result['MAE_test']/y_original.mean()*100
overfit_pct = (best_result['MAE_test'] / best_result['MAE_train'] - 1) * 100

print(f"\n📈 性能评价:")
if relative_error_pct < 15:
    print(f"  ✅ 预测精度: 🎉 优秀！相对误差 {relative_error_pct:.1f}% < 15%")
elif relative_error_pct < 20:
    print(f"  ✅ 预测精度: 良好，相对误差 {relative_error_pct:.1f}% < 20%")
else:
    print(f"  ⚠️ 预测精度: 一般，相对误差 {relative_error_pct:.1f}% > 20%，建议优化")

if abs(overfit_pct) < 10:
    print(f"  ✅ 泛化能力: 优秀，过拟合程度 {overfit_pct:+.1f}% < 10%")
elif abs(overfit_pct) < 20:
    print(f"  ✅ 泛化能力: 良好，过拟合程度 {overfit_pct:+.1f}% < 20%")
else:
    print(f"  ⚠️ 泛化能力: 有过拟合，过拟合程度 {overfit_pct:+.1f}% > 20%")

if best_result['R2_test'] > 0.7:
    print(f"  ✅ 拟合优度: 优秀，R² = {best_result['R2_test']:.4f} > 0.7")
elif best_result['R2_test'] > 0.6:
    print(f"  ✅ 拟合优度: 良好，R² = {best_result['R2_test']:.4f} > 0.6")
else:
    print(f"  ⚠️ 拟合优度: 一般，R² = {best_result['R2_test']:.4f} < 0.6")

print(f"\n💡 使用说明:")
print("  1. 预测时需要先对新数据用scaler标准化")
print("  2. 模型输出是log(月租+1)，需要用np.expm1()转回原始月租")
print("  3. 示例代码:")
print("     model = joblib.load('rent_best_model_log_月租.pkl')")
print("     scaler = joblib.load('rent_scaler_log_月租.pkl')")
print("     features = joblib.load('rent_selected_features_log_月租.pkl')")
print("     X_new_scaled = scaler.transform(X_new[features])")
print("     y_pred_log = model.predict(X_new_scaled)")
print("     y_pred = np.expm1(y_pred_log)  # 转回原始月租")

print("\n" + "=" * 80)
print("✨ 建模完成！✨")
print("=" * 80)

租房价格预测 - 对数月租线性回归建模
🎯 改进：根据is_train划分 + 对数变换 + 训练集异常值清理

Step 1: 数据加载 + 数据清洗
原始数据形状: (97709, 305)

--------------------------------------------------------------------------------
1.0 清洗面积列（去除单位）
--------------------------------------------------------------------------------
找到 2 个面积列: ['面积', '面积.1']
  面积 已是数值类型，跳过
  面积.1 已是数值类型，跳过

删除 1 个城市特征: ['城市']

确认的连续特征(7个): ['面积', '容积率', '室', '房屋总数', '楼栋总数', '绿化率', '物业费']

--------------------------------------------------------------------------------
1.1 计算Price（月租 × 付款方式，仅用于展示）
--------------------------------------------------------------------------------
Price计算完成（仅用于分析）:
  月租范围: [2903, 15404193]
  Price范围: [17938, 46212579]

--------------------------------------------------------------------------------
1.2 对月租取对数（处理右偏分布）
--------------------------------------------------------------------------------
原始月租统计:
  均值: 561537.71
  标准差: 625523.86
  范围: [2903.26, 15404193.15]
  偏度: 5.00 (>1为右偏)

对数月租统计:
  均值: 12.7892
  标准差: 1.0377
  范围: [7.9

In [26]:

import pandas as pd

# 定义需要保留的列
keep_columns = [
    '城市', '户型',  'Price', '楼层', '面积', '朝向', 
    '交易时间', '付款方式', '租赁方式', '电梯', '配套设施', 
    'lon', 'lat', '板块', '建筑年代', '房屋总数', '楼栋总数', 
    '绿 化 率', '容 积 率', '物 业 费'  
]

# 读取预测集数据（尝试不同编码）
try:
    test_df = pd.read_csv('ruc_Class25Q2_test_rent.csv', encoding='utf-8')
    print("使用 utf-8 编码成功")
except:
    try:
        test_df = pd.read_csv('ruc_Class25Q2_test_rent.csv', encoding='gbk')
        print("使用 gbk 编码成功")
    except:
        test_df = pd.read_csv('ruc_Class25Q2_test_rent.csv', encoding='gb2312')
        print("使用 gb2312 编码成功")

print(f"\n原始预测集: {test_df.shape}")
print(f"列名: {list(test_df.columns)}")

# 只保留指定的列（排除Price，因为预测集没有）
test_keep_cols = [col for col in keep_columns if col in test_df.columns and col != 'Price']
test_df = test_df[test_keep_cols]

print(f"\n处理后预测集: {test_df.shape}")
print(f"保留的列: {list(test_df.columns)}")

# 保存为新文件（使用utf-8编码）
test_df.to_csv('ruc_Class25Q2_test_rent1.csv', index=False, encoding='utf-8-sig')
print("\n新文件已保存: ruc_Class25Q2_test_rent1.csv")


使用 utf-8 编码成功

原始预测集: (9773, 46)
列名: ['ID', '城市', '户型', '装修', '楼层', '面积', '朝向', '交易时间', '付款方式', '租赁方式', '电梯', '车位', '用水', '用电', '燃气', '采暖', '租期', '配套设施', 'lon', 'lat', '年份', '区县', '板块', '环线位置', '物业类别', '建筑年代', '开发商', '房屋总数', '楼栋总数', '物业公司', '绿 化 率', '容 积 率', '物 业 费', '建筑结构', '物业办公电话', '产权描述', '供水', '供暖', '供电', '燃气费', '供热费', '停车位', '停车费用', 'coord_x', 'coord_y', '客户反馈']

处理后预测集: (9773, 19)
保留的列: ['城市', '户型', '楼层', '面积', '朝向', '交易时间', '付款方式', '租赁方式', '电梯', '配套设施', 'lon', 'lat', '板块', '建筑年代', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费']

新文件已保存: ruc_Class25Q2_test_rent1.csv


In [20]:

import pandas as pd
import numpy as np

# 读取数据
test_df = pd.read_csv('ruc_Class25Q2_test_rent1.csv')

print(f"原始数据: {test_df.shape}")
print(f"\n户型样例: \n{test_df['户型'].head(10)}")
print(f"\n面积原始样例: \n{test_df['面积'].head(10)}")

# ========== 1. 面积去掉单位 ==========
# 去掉㎡、平方米、平米等单位，只保留数字
test_df['面积_temp'] = test_df['面积'].astype(str).str.replace('㎡', '').str.replace('平方米', '').str.replace('平米', '').str.replace('平', '').str.strip()
test_df['面积_temp'] = pd.to_numeric(test_df['面积_temp'], errors='coerce')

print(f"\n去掉单位后面积: \n{test_df['面积_temp'].describe()}")

# ========== 2. 户型清洗 ==========
# 提取室数、厅数、房间数
test_df['室数_temp'] = test_df['户型'].str.extract(r'(\d+)室').astype(float)
test_df['厅数_temp'] = test_df['户型'].str.extract(r'(\d+)厅').astype(float)
test_df['房间数_temp'] = test_df['户型'].str.extract(r'(\d+)房间').astype(float)

# 计算"室"列
# 如果有"室"和"厅"，室 = 室数 + 厅数
# 如果只有"房间"，室 = 房间数
test_df['室'] = np.nan

# 优先使用 室+厅
mask_室厅 = test_df['室数_temp'].notna()
test_df.loc[mask_室厅, '室'] = test_df.loc[mask_室厅, '室数_temp'] + test_df.loc[mask_室厅, '厅数_temp'].fillna(0)

# 其次使用 房间数
mask_房间 = (test_df['室'].isna()) & (test_df['房间数_temp'].notna())
test_df.loc[mask_房间, '室'] = test_df.loc[mask_房间, '房间数_temp']

# 缺失值用4填补
test_df['室'].fillna(4, inplace=True)

print(f"\n室统计: \n{test_df['室'].describe()}")

# ========== 3. 面积Winsorize处理 ==========
from scipy.stats import mstats

# 进行winsorize处理（默认上下各5%）
test_df['面积'] = mstats.winsorize(test_df['面积_temp'].dropna(), limits=[0.05, 0.05])

print(f"\nWinsorize后面积: \n{test_df['面积'].describe()}")

# ========== 4. 删除原户型列和中间列 ==========
test_df.drop(columns=['户型', '面积_temp', '室数_temp', '厅数_temp', '房间数_temp'], inplace=True)

print(f"\n最终保留的列: {list(test_df.columns)}")

# 保存处理后的数据
test_df.to_csv('ruc_Class25Q2_test_rent2.csv', index=False, encoding='utf-8-sig')
print("\n清洗后数据已保存: ruc_Class25Q2_test_rent2.csv")


原始数据: (9773, 19)

户型样例: 
0    2室2厅1卫
1    2室1厅1卫
2    2室2厅1卫
3    2室1厅1卫
4    3室2厅2卫
5    3室2厅1卫
6    3室2厅2卫
7    3室1厅1卫
8    2室1厅1卫
9    1室1厅1卫
Name: 户型, dtype: object

面积原始样例: 
0     86.94㎡
1     72.60㎡
2     98.00㎡
3     98.97㎡
4    170.53㎡
5     86.77㎡
6    135.00㎡
7     52.09㎡
8    106.50㎡
9     63.00㎡
Name: 面积, dtype: object

去掉单位后面积: 
count    9773.000000
mean       81.313054
std        42.321462
min         7.000000
25%        50.400000
50%        80.000000
75%       101.000000
max       380.220000
Name: 面积_temp, dtype: float64

室统计: 
count    9773.000000
mean        3.712371
std         1.409737
min         1.000000
25%         3.000000
50%         4.000000
75%         5.000000
max        10.000000
Name: 室, dtype: float64

Winsorize后面积: 
count    9773.000000
mean       79.415407
std        35.115580
min        20.000000
25%        50.400000
50%        80.000000
75%       101.000000
max       147.710000
Name: 面积, dtype: float64

最终保留的列: ['城市', '楼层', '面积', '朝向', '交易时间', '付款方式', 

In [8]:

import pandas as pd
import numpy as np

# 读取数据
test_df = pd.read_csv('ruc_Class25Q2_test_rent2.csv')

print(f"原始数据: {test_df.shape}")
print(f"\n付款方式缺失情况:")
print(test_df['付款方式'].value_counts(dropna=False))

# ========== 1. 建立城市到付款方式众数的映射 ==========
city_payment_mode = {
    0: '季付价',
    1: '季付价',
    2: '季付价',
    3: '季付价',
    4: '季付价',
    5: '月付价',
    6: '年付价',
    7: '月付价',
    8: '半年付价',
    9: '季付价',
    10: '月付价',
    11: '季付价'
}

# ========== 2. 按城市填补付款方式缺失值 ==========
def fill_payment_by_city(row):
    if pd.isna(row['付款方式']):
        city = row['城市']
        return city_payment_mode.get(city, '季付价')  # 默认用季付价
    else:
        return row['付款方式']

test_df['付款方式'] = test_df.apply(fill_payment_by_city, axis=1)

print(f"\n填补后付款方式分布:")
print(test_df['付款方式'].value_counts())

# ========== 3. One-hot编码（与训练集保持一致，使用0和1）==========
payment_dummies = pd.get_dummies(test_df['付款方式'], prefix='付款方式', drop_first=False)

# 转换为0和1（整数）
payment_dummies = payment_dummies.astype(int)

print(f"\n所有One-hot编码列: {list(payment_dummies.columns)}")

# 训练集中存在的付款方式列
train_payment_cols = ['付款方式_双月付价', '付款方式_季付价', '付款方式_年付价', '付款方式_月付价']

# 确保测试集有训练集的所有列（如果没有则填充0）
for col in train_payment_cols:
    if col not in payment_dummies.columns:
        payment_dummies[col] = 0
        print(f"添加缺失列: {col}")

# 只保留训练集中存在的列
payment_dummies = payment_dummies[train_payment_cols]

print(f"\n保留的One-hot编码列: {list(payment_dummies.columns)}")
print(f"\n编码示例（前5行）:")
print(payment_dummies.head())

# ========== 4. 合并到原数据并删除原列 ==========
test_df = pd.concat([test_df, payment_dummies], axis=1)
test_df.drop(columns=['付款方式'], inplace=True)

print(f"\n最终数据: {test_df.shape}")
print(f"最终列名: {list(test_df.columns)}")

# 保存处理后的数据
test_df.to_csv('ruc_Class25Q2_test_rent3.csv', index=False, encoding='utf-8-sig')
print("\n处理后数据已保存: ruc_Class25Q2_test_rent3.csv")


原始数据: (9773, 19)

付款方式缺失情况:
付款方式
季付价     5325
NaN     2387
月付价     1726
半年付价     284
年付价       49
双月付价       2
Name: count, dtype: int64

填补后付款方式分布:
付款方式
季付价     6537
月付价     2317
半年付价     695
年付价      222
双月付价       2
Name: count, dtype: int64

所有One-hot编码列: ['付款方式_半年付价', '付款方式_双月付价', '付款方式_季付价', '付款方式_年付价', '付款方式_月付价']

保留的One-hot编码列: ['付款方式_双月付价', '付款方式_季付价', '付款方式_年付价', '付款方式_月付价']

编码示例（前5行）:
   付款方式_双月付价  付款方式_季付价  付款方式_年付价  付款方式_月付价
0          0         1         0         0
1          0         0         0         1
2          0         1         0         0
3          0         1         0         0
4          0         1         0         0

最终数据: (9773, 22)
最终列名: ['城市', '楼层', '面积', '朝向', '交易时间', '租赁方式', '电梯', '配套设施', 'lon', 'lat', '板块', '建筑年代', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', '室', '付款方式_双月付价', '付款方式_季付价', '付款方式_年付价', '付款方式_月付价']

处理后数据已保存: ruc_Class25Q2_test_rent3.csv


In [34]:

import pandas as pd
import numpy as np
import re

# 读取数据
test_df = pd.read_csv('ruc_Class25Q2_test_rent3.csv')

print(f"原始数据: {test_df.shape}")

# ========== 1. 朝向处理：提取东南西北四个方向 ==========
print(f"\n朝向样例:")
print(test_df['朝向'].head(10))

def extract_directions(direction_str):
    """提取朝向中的东南西北"""
    if pd.isna(direction_str):
        return {'东': 0, '南': 0, '西': 0, '北': 0}
    
    direction_str = str(direction_str).replace(' ', '')  # 去除空格
    return {
        '东': 1 if '东' in direction_str else 0,
        '南': 1 if '南' in direction_str else 0,
        '西': 1 if '西' in direction_str else 0,
        '北': 1 if '北' in direction_str else 0
    }

# 应用函数
direction_features = test_df['朝向'].apply(extract_directions).apply(pd.Series)
direction_features.columns = ['朝向_东', '朝向_南', '朝向_西', '朝向_北']

print(f"\n朝向特征提取结果（前5行）:")
print(direction_features.head())

# 合并到原数据
test_df = pd.concat([test_df, direction_features], axis=1)
test_df.drop(columns=['朝向'], inplace=True)

# ========== 2. 交易时间处理：提取年和月，进行one-hot编码 ==========
print(f"\n交易时间样例:")
print(test_df['交易时间'].head(10))

# 提取年和月（格式：2023/2/1）
test_df['年'] = pd.to_datetime(test_df['交易时间'], errors='coerce').dt.year
test_df['月'] = pd.to_datetime(test_df['交易时间'], errors='coerce').dt.month

print(f"\n提取的年: {sorted(test_df['年'].dropna().unique())}")
print(f"提取的月: {sorted(test_df['月'].dropna().unique())}")

# 年的one-hot编码
year_dummies = pd.get_dummies(test_df['年'], prefix='年').astype(int)
print(f"测试集年份列: {list(year_dummies.columns)}")

# 确保有训练集的年_2025列
if '年_2025' not in year_dummies.columns:
    year_dummies['年_2025'] = 0
    print("添加缺失列: 年_2025")

# 只保留训练集存在的年_2025列
year_dummies = year_dummies[['年_2025']]

# 月的one-hot编码
month_dummies = pd.get_dummies(test_df['月'], prefix='月').astype(int)
print(f"测试集月份列: {list(month_dummies.columns)}")

# 训练集存在的月份列
train_month_cols = ['月_1', '月_2', '月_3', '月_4', '月_6', '月_7', '月_8', '月_9', '月_10', '月_11', '月_12']
for col in train_month_cols:
    if col not in month_dummies.columns:
        month_dummies[col] = 0
        print(f"添加缺失列: {col}")

# 只保留训练集存在的列
month_dummies = month_dummies[train_month_cols]

print(f"\n最终年份one-hot列: {list(year_dummies.columns)}")
print(f"最终月份one-hot列: {list(month_dummies.columns)}")

# 合并到原数据
test_df = pd.concat([test_df, year_dummies, month_dummies], axis=1)
test_df.drop(columns=['交易时间', '年', '月'], inplace=True)

# ========== 3. 建筑年代处理：提取年份，计算房龄 ==========
print(f"\n建筑年代样例:")
print(test_df['建筑年代'].head(10))

def extract_build_year(year_str):
    """提取建筑年代，区间取最后一年"""
    if pd.isna(year_str):
        return np.nan
    
    year_str = str(year_str)
    # 提取所有数字（4位年份）
    years = re.findall(r'\d{4}', year_str)
    if years:
        return int(max(years))  # 取最大值（最后一年）
    return np.nan

test_df['建筑年代_提取'] = test_df['建筑年代'].apply(extract_build_year)

print(f"\n建筑年代提取结果:")
print(test_df['建筑年代_提取'].describe())
print(f"缺失值数量: {test_df['建筑年代_提取'].isna().sum()}")

# 缺失值填充房龄中位数12对应的建筑年代
median_age = 12
current_year = 2025
median_build_year = current_year - median_age  # 2013

test_df['建筑年代_提取'].fillna(median_build_year, inplace=True)

# 计算房龄
test_df['房龄'] = current_year - test_df['建筑年代_提取']

print(f"\n房龄统计:")
print(test_df['房龄'].describe())

# 删除原列和中间列
test_df.drop(columns=['建筑年代', '建筑年代_提取'], inplace=True)

# ========== 4. 保存结果 ==========
print(f"\n最终数据: {test_df.shape}")
print(f"最终列名: {list(test_df.columns)}")

test_df.to_csv('ruc_Class25Q2_test_rent4.csv', index=False, encoding='utf-8-sig')
print("\n处理后数据已保存: ruc_Class25Q2_test_rent4.csv")


原始数据: (9773, 22)

朝向样例:
0    南 北
1      南
2      南
3    东 西
4      南
5      南
6    南 北
7      南
8      南
9      南
Name: 朝向, dtype: object

朝向特征提取结果（前5行）:
   朝向_东  朝向_南  朝向_西  朝向_北
0     0     1     0     1
1     0     1     0     0
2     0     1     0     0
3     1     0     1     0
4     0     1     0     0

交易时间样例:
0    2025-08-01
1    2025-05-23
2    2025-02-18
3    2025-02-17
4    2025-03-24
5    2025-05-13
6    2025-04-14
7    2025-04-29
8    2025-09-20
9    2025-05-09
Name: 交易时间, dtype: object

提取的年: [np.int32(2025)]
提取的月: [np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11)]
测试集年份列: ['年_2025']
测试集月份列: ['月_2', '月_3', '月_4', '月_5', '月_6', '月_7', '月_8', '月_9', '月_10', '月_11']
添加缺失列: 月_1
添加缺失列: 月_12

最终年份one-hot列: ['年_2025']
最终月份one-hot列: ['月_1', '月_2', '月_3', '月_4', '月_6', '月_7', '月_8', '月_9', '月_10', '月_11', '月_12']

建筑年代样例:
0    2011-2019年
1    1985-2008年
2           NaN
3    2004-2009年
4           NaN

In [50]:

import pandas as pd
import numpy as np
import re

# 读取数据
test_df = pd.read_csv('ruc_Class25Q2_test_rent4.csv')

print(f"原始数据: {test_df.shape}")

# ========== 1. 楼层处理 ==========
print(f"\n楼层样例:")
print(test_df['楼层'].value_counts().head(20))

def parse_floor(floor_str):
    """
    解析楼层信息
    返回: (当前楼层, 总楼层数)
    """
    if pd.isna(floor_str):
        return None, None
    
    floor_str = str(floor_str).strip()
    
    # 地下室
    if '地下' in floor_str or '地库' in floor_str:
        return 'underground', None
    
    # 格式1: 中楼层/23层
    if '低楼层' in floor_str or '中楼层' in floor_str or '高楼层' in floor_str:
        # 提取总楼层数
        match = re.search(r'/(\d+)层', floor_str)
        if match:
            total_floors = int(match.group(1))
            if '低楼层' in floor_str:
                return 'low', total_floors
            elif '中楼层' in floor_str:
                return 'mid', total_floors
            elif '高楼层' in floor_str:
                return 'high', total_floors
        return None, None
    
    # 格式2: 30/45层
    match = re.search(r'(\d+)/(\d+)', floor_str)
    if match:
        current_floor = int(match.group(1))
        total_floors = int(match.group(2))
        return current_floor, total_floors
    
    return None, None

# 应用解析函数
test_df[['当前楼层', '总楼层']] = test_df['楼层'].apply(lambda x: pd.Series(parse_floor(x)))

print(f"\n解析结果:")
print(f"当前楼层样例: {test_df['当前楼层'].value_counts().head(10)}")
print(f"总楼层样例: {test_df['总楼层'].value_counts().head(10)}")

# 分类楼层
def classify_floor(row):
    """将楼层分为：低层、中层、高层、地下室"""
    current = row['当前楼层']
    total = row['总楼层']
    
    if current == 'underground':
        return '地下室'
    elif current == 'low':
        return '低层'
    elif current == 'mid':
        return '中层'
    elif current == 'high':
        return '高层'
    elif isinstance(current, (int, float)) and isinstance(total, (int, float)):
        ratio = current / total
        if ratio <= 1/3:
            return '低层'
        elif ratio <= 2/3:
            return '中层'
        else:
            return '高层'
    else:
        return '中层'  # 缺失值默认中层

test_df['楼层分类'] = test_df.apply(classify_floor, axis=1)

print(f"\n楼层分类分布:")
print(test_df['楼层分类'].value_counts())

# One-hot编码
floor_dummies = pd.get_dummies(test_df['楼层分类'], prefix='楼层').astype(int)

# 训练集存在的列
train_floor_cols = ['楼层_中层', '楼层_高层', '楼层_地下室']

# 确保有训练集的所有列
for col in train_floor_cols:
    if col not in floor_dummies.columns:
        floor_dummies[col] = 0
        print(f"添加缺失列: {col}")

# 只保留训练集存在的列（低层作为基准）
floor_dummies = floor_dummies[train_floor_cols]

print(f"\n楼层One-hot列: {list(floor_dummies.columns)}")

# 合并到原数据
test_df = pd.concat([test_df, floor_dummies], axis=1)

# ========== 2. 电梯处理 ==========
print(f"\n电梯样例:")
print(test_df['电梯'].value_counts(dropna=False))

# 填充缺失值：总楼层>6则有电梯，否则无电梯
def fill_elevator(row):
    if pd.notna(row['电梯']):
        return row['电梯']
    else:
        # 根据总楼层判断
        if pd.notna(row['总楼层']) and row['总楼层'] > 6:
            return '有'
        else:
            return '无'

test_df['电梯'] = test_df.apply(fill_elevator, axis=1)

print(f"\n填补后电梯分布:")
print(test_df['电梯'].value_counts())

# One-hot编码（以"无"为基准）
elevator_dummies = pd.get_dummies(test_df['电梯'], prefix='电梯').astype(int)

# 删除"电梯_无"列（作为基准）
if '电梯_无' in elevator_dummies.columns:
    elevator_dummies = elevator_dummies.drop(columns=['电梯_无'])
    print(f"\n已删除'电梯_无'作为参考类别")

# 确保有"电梯_有"列
if '电梯_有' not in elevator_dummies.columns:
    elevator_dummies['电梯_有'] = 0

print(f"\n电梯One-hot列: {list(elevator_dummies.columns)}")

# 合并到原数据
test_df = pd.concat([test_df, elevator_dummies], axis=1)

# ========== 3. 删除原列和中间列 ==========
test_df.drop(columns=['楼层', '当前楼层', '总楼层', '楼层分类', '电梯'], inplace=True)

print(f"\n最终数据: {test_df.shape}")
print(f"最终列名: {list(test_df.columns)}")

# 保存处理后的数据
test_df.to_csv('ruc_Class25Q2_test_rent5.csv', index=False, encoding='utf-8-sig')
print("\n处理后数据已保存: ruc_Class25Q2_test_rent5.csv")


原始数据: (9773, 36)

楼层样例:
楼层
高楼层/6层     508
中楼层/6层     327
高楼层/18层    239
低楼层/6层     226
低楼层/18层    219
中楼层/18层    198
中楼层/5层     183
中楼层/7层     172
中楼层/32层    168
高楼层/33层    164
高楼层/7层     160
中楼层/11层    154
中楼层/33层    146
低楼层/33层    139
高楼层/32层    136
中楼层/8层     120
低楼层/32层    113
低楼层/34层    104
中楼层/34层    101
中楼层/26层     90
Name: count, dtype: int64

解析结果:
当前楼层样例: 当前楼层
mid     3129
high    2780
low     2380
6        157
5        153
4        138
3        129
2         96
7         77
1         69
Name: count, dtype: int64
总楼层样例: 总楼层
6.0     1371
18.0     767
33.0     513
32.0     482
7.0      479
5.0      335
11.0     333
34.0     299
27.0     289
28.0     280
Name: count, dtype: int64

楼层分类分布:
楼层分类
中层     3628
高层     3314
低层     2791
地下室      40
Name: count, dtype: int64

楼层One-hot列: ['楼层_中层', '楼层_高层', '楼层_地下室']

电梯样例:
电梯
有    6496
无    3277
Name: count, dtype: int64

填补后电梯分布:
电梯
有    6496
无    3277
Name: count, dtype: int64

已删除'电梯_无'作为参考类别

电梯One-hot列: ['电梯_有']

最终数据: (9773, 38)
最终

In [37]:

import pandas as pd
import numpy as np
import re

# 读取数据
test_df = pd.read_csv('ruc_Class25Q2_test_rent5.csv')

print(f"原始数据: {test_df.shape}")
print(f"\n所有列名: {list(test_df.columns)}")

# ========== 1. 房屋总数处理 ==========
print(f"\n房屋总数样例:")
print(test_df['房屋总数'].head(10))

def extract_number(value):
    """提取数字，去除单位"""
    if pd.isna(value):
        return np.nan
    
    value_str = str(value).strip()
    # 提取数字（包括小数）
    match = re.search(r'(\d+\.?\d*)', value_str)
    if match:
        return float(match.group(1))
    return np.nan

test_df['房屋总数'] = test_df['房屋总数'].apply(extract_number)
test_df['房屋总数'].fillna(1445.00, inplace=True)

print(f"房屋总数处理后: {test_df['房屋总数'].describe()}")

# ========== 2. 楼栋总数处理 ==========
print(f"\n楼栋总数样例:")
print(test_df['楼栋总数'].head(10))

test_df['楼栋总数'] = test_df['楼栋总数'].apply(extract_number)
test_df['楼栋总数'].fillna(13.00, inplace=True)

print(f"楼栋总数处理后: {test_df['楼栋总数'].describe()}")

# ========== 3. 绿化率处理（注意列名可能有空格）==========
# 查找绿化率列名（可能是"绿化率"或"绿 化 率"）
greenrate_col = None
for col in test_df.columns:
    if '绿' in col and '化' in col and '率' in col:
        greenrate_col = col
        break

if greenrate_col is None:
    print("\n警告：未找到绿化率列！")
else:
    print(f"\n找到绿化率列: '{greenrate_col}'")
    print(test_df[greenrate_col].head(10))

    def extract_rate(value):
        """提取百分比，去除%符号"""
        if pd.isna(value):
            return np.nan
        
        value_str = str(value).strip().replace('%', '')
        # 提取数字（包括小数）
        match = re.search(r'(\d+\.?\d*)', value_str)
        if match:
            return float(match.group(1))
        return np.nan

    test_df[greenrate_col] = test_df[greenrate_col].apply(extract_rate)
    test_df[greenrate_col].fillna(35.00, inplace=True)
    
    # 重命名为标准列名（去除空格）
    test_df.rename(columns={greenrate_col: '绿化率'}, inplace=True)
    
    print(f"绿化率处理后: {test_df['绿化率'].describe()}")

# ========== 4. 容积率处理（注意列名可能有空格）==========
# 查找容积率列名
volumerate_col = None
for col in test_df.columns:
    if '容' in col and '积' in col and '率' in col:
        volumerate_col = col
        break

if volumerate_col is None:
    print("\n警告：未找到容积率列！")
else:
    print(f"\n找到容积率列: '{volumerate_col}'")
    print(test_df[volumerate_col].head(10))

    test_df[volumerate_col] = test_df[volumerate_col].apply(extract_number)
    test_df[volumerate_col].fillna(2.67, inplace=True)
    
    # 重命名为标准列名（去除空格）
    test_df.rename(columns={volumerate_col: '容积率'}, inplace=True)
    
    print(f"容积率处理后: {test_df['容积率'].describe()}")

# ========== 5. 物业费处理（可能是区间，取均值）==========
# 查找物业费列名
propertyfee_col = None
for col in test_df.columns:
    if '物' in col and '业' in col and '费' in col:
        propertyfee_col = col
        break

if propertyfee_col is None:
    print("\n警告：未找到物业费列！")
else:
    print(f"\n找到物业费列: '{propertyfee_col}'")
    print(test_df[propertyfee_col].head(10))

    def extract_property_fee(value):
        """提取物业费，区间取均值"""
        if pd.isna(value):
            return np.nan
        
        value_str = str(value).strip().replace('元/平米/月', '').replace('元', '').strip()
        
        # 检查是否是区间格式（如"2.0-3.0"）
        if '-' in value_str or '~' in value_str:
            # 提取两个数字
            numbers = re.findall(r'(\d+\.?\d*)', value_str)
            if len(numbers) >= 2:
                return (float(numbers[0]) + float(numbers[1])) / 2
            elif len(numbers) == 1:
                return float(numbers[0])
        else:
            # 提取单个数字
            match = re.search(r'(\d+\.?\d*)', value_str)
            if match:
                return float(match.group(1))
        
        return np.nan

    test_df[propertyfee_col] = test_df[propertyfee_col].apply(extract_property_fee)
    test_df[propertyfee_col].fillna(2.20, inplace=True)
    
    # 重命名为标准列名（去除空格）
    test_df.rename(columns={propertyfee_col: '物业费'}, inplace=True)
    
    print(f"物业费处理后: {test_df['物业费'].describe()}")

# ========== 6. 租赁方式处理（one-hot编码）==========
print(f"\n租赁方式样例:")
print(test_df['租赁方式'].value_counts(dropna=False))

# One-hot编码
lease_dummies = pd.get_dummies(test_df['租赁方式'], prefix='租赁方式').astype(int)

print(f"租赁方式One-hot列: {list(lease_dummies.columns)}")

# 只保留"租赁方式_整租"列（合租作为基准）
if '租赁方式_整租' not in lease_dummies.columns:
    lease_dummies['租赁方式_整租'] = 0
    print("添加缺失列: 租赁方式_整租")

lease_dummies = lease_dummies[['租赁方式_整租']]

print(f"保留的租赁方式列: {list(lease_dummies.columns)}")

# 合并到原数据
test_df = pd.concat([test_df, lease_dummies], axis=1)
test_df.drop(columns=['租赁方式'], inplace=True)

# ========== 7. 保存结果 ==========
print(f"\n最终数据: {test_df.shape}")
print(f"最终列名: {list(test_df.columns)}")

test_df.to_csv('ruc_Class25Q2_test_rent6.csv', index=False, encoding='utf-8-sig')
print("\n处理后数据已保存: ruc_Class25Q2_test_rent6.csv")


原始数据: (9773, 38)

所有列名: ['城市', '面积', '租赁方式', '配套设施', 'lon', 'lat', '板块', '房屋总数', '楼栋总数', '绿 化 率', '容 积 率', '物 业 费', '室', '付款方式_双月付价', '付款方式_季付价', '付款方式_年付价', '付款方式_月付价', '朝向_东', '朝向_南', '朝向_西', '朝向_北', '年_2025', '月_1', '月_2', '月_3', '月_4', '月_6', '月_7', '月_8', '月_9', '月_10', '月_11', '月_12', '房龄', '楼层_中层', '楼层_高层', '楼层_地下室', '电梯_有']

房屋总数样例:
0     992户
1     806户
2    1327户
3    2660户
4    1196户
5    3253户
6      NaN
7    2088户
8      NaN
9    3445户
Name: 房屋总数, dtype: object
房屋总数处理后: count     9773.000000
mean      1841.377571
std       1665.620786
min          1.000000
25%        794.000000
50%       1445.000000
75%       2272.000000
max      12669.000000
Name: 房屋总数, dtype: float64

楼栋总数样例:
0    12栋
1    15栋
2    13栋
3    19栋
4    20栋
5    12栋
6    NaN
7    85栋
8    NaN
9    65栋
Name: 楼栋总数, dtype: object
楼栋总数处理后: count    9773.000000
mean       27.025069
std        52.133685
min         1.000000
25%         7.000000
50%        13.000000
75%        26.000000
max       734.000000
Name: 楼

In [38]:

import pandas as pd
import numpy as np

# 读取数据
test_df = pd.read_csv('ruc_Class25Q2_test_rent6.csv')

print(f"原始数据: {test_df.shape}")

# ========== 配套设施处理 ==========
print(f"\n配套设施样例:")
print(test_df['配套设施'].head(10))

# 定义需要提取的物品
facilities = ['床', '衣柜', '空调', '洗衣机', '热水器', '冰箱', '天然气', '电视', '暖气', '宽带']

# 高频物品（>=50%覆盖率），缺失值填1
high_freq_facilities = ['床', '衣柜', '空调', '洗衣机', '热水器']

def extract_facilities(facility_str):
    """提取配套设施中的各个物品"""
    result = {}
    
    if pd.isna(facility_str):
        # 缺失值处理
        for item in facilities:
            if item in high_freq_facilities:
                result[item] = 1  # 高频物品默认有
            else:
                result[item] = 0  # 低频物品默认无
        return result
    
    facility_str = str(facility_str)
    
    for item in facilities:
        if item == '天然气':
            # 特殊处理：只要包含"天然"就认定为有天然气
            result[item] = 1 if '天然' in facility_str else 0
        else:
            result[item] = 1 if item in facility_str else 0
    
    return result

# 应用函数提取所有设施
facility_features = test_df['配套设施'].apply(extract_facilities).apply(pd.Series)

# 重命名列，加上"设施_"前缀
facility_features.columns = [f'设施_{col}' for col in facility_features.columns]

print(f"\n提取的设施特征:")
print(facility_features.head(10))

print(f"\n各设施覆盖率:")
for col in facility_features.columns:
    coverage = (facility_features[col] == 1).sum()
    percentage = coverage / len(facility_features) * 100
    print(f"  {col}: {coverage} 条记录 ({percentage:.1f}%)")

# 确保列的顺序与训练集一致
train_facility_cols = ['设施_床', '设施_衣柜', '设施_空调', '设施_洗衣机', '设施_热水器', 
                       '设施_冰箱', '设施_天然气', '设施_电视', '设施_暖气', '设施_宽带']

# 检查并添加缺失列
for col in train_facility_cols:
    if col not in facility_features.columns:
        facility_features[col] = 0
        print(f"添加缺失列: {col}")

# 按训练集顺序重排列
facility_features = facility_features[train_facility_cols]

print(f"\n最终设施列: {list(facility_features.columns)}")

# 合并到原数据
test_df = pd.concat([test_df, facility_features], axis=1)
test_df.drop(columns=['配套设施'], inplace=True)

# ========== 保存结果 ==========
print(f"\n最终数据: {test_df.shape}")
print(f"最终列名: {list(test_df.columns)}")

test_df.to_csv('ruc_Class25Q2_test_rent7.csv', index=False, encoding='utf-8-sig')
print("\n处理后数据已保存: ruc_Class25Q2_test_rent7.csv")


原始数据: (9773, 38)

配套设施样例:
0    洗衣机、空调、衣柜、电视、冰箱、热水器、床、暖气、宽带、天然
1                洗衣机、空调、衣柜、冰箱、热水器、床
2         洗衣机、空调、衣柜、电视、冰箱、热水器、床、天然气
3                               NaN
4         洗衣机、空调、衣柜、电视、冰箱、热水器、床、天然气
5    洗衣机、空调、衣柜、电视、冰箱、热水器、床、暖气、宽带、天然
6                               NaN
7                         空调、冰箱、热水器
8                               NaN
9            洗衣机、空调、衣柜、冰箱、热水器、床、天然气
Name: 配套设施, dtype: object

提取的设施特征:
   设施_床  设施_衣柜  设施_空调  设施_洗衣机  设施_热水器  设施_冰箱  设施_天然气  设施_电视  设施_暖气  设施_宽带
0     1      1      1       1       1      1       1      1      1      1
1     1      1      1       1       1      1       0      0      0      0
2     1      1      1       1       1      1       1      1      0      0
3     1      1      1       1       1      0       0      0      0      0
4     1      1      1       1       1      1       1      1      0      0
5     1      1      1       1       1      1       1      1      1      1
6     1      1      1       1       1      0       0      0 

In [ ]:

# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
from scipy.spatial.distance import cdist

# 读取数据
test_path = "ruc_Class25Q2_test_rent7.csv"
train_path = "ruc_Class25Q2_train_rent91.csv"

df_test = pd.read_csv(test_path, encoding='utf-8')
df_train = pd.read_csv(train_path, encoding='utf-8')

print(f"Test数据: {df_test.shape}")
print(f"Train数据: {df_train.shape}")

# 找到所有聚类变量列
cluster_cols = [col for col in df_train.columns if col.startswith('cluster_')]
print(f"\n找到 {len(cluster_cols)} 个聚类变量")
print(f"聚类变量范围: {cluster_cols[0]} 到 {cluster_cols[-1]}")

# 确认城市列和经纬度列
city_col = '城市'
lon_col = 'lon'
lat_col = 'lat'

# 检查列是否存在
if city_col not in df_test.columns or city_col not in df_train.columns:
    print(f"\n错误：未找到'{city_col}'列！")
    print(f"Test列名: {list(df_test.columns[:20])}")
    print(f"Train列名: {list(df_train.columns[:20])}")
else:
    print(f"\n城市列: {city_col}")
    print(f"Test城市分布:\n{df_test[city_col].value_counts().sort_index()}")

if lon_col not in df_test.columns or lat_col not in df_test.columns:
    print(f"\n错误：未找到经纬度列！")
    print(f"Test列名: {list(df_test.columns)}")
else:
    print(f"\n经纬度列: {lon_col}, {lat_col}")

# 初始化聚类变量列（全部设为0）
for col in cluster_cols:
    df_test[col] = 0

# 按城市匹配
unique_cities = sorted(df_test[city_col].dropna().unique())
print(f"\n开始匹配，共 {len(unique_cities)} 个城市...\n")

total_matched = 0

for city in unique_cities:
    # 筛选该城市的test和train数据（同时过滤经纬度缺失）
    test_city_idx = (df_test[city_col] == city) & \
                    df_test[lon_col].notna() & \
                    df_test[lat_col].notna()
    
    train_city = df_train[(df_train[city_col] == city) & 
                          df_train[lon_col].notna() & 
                          df_train[lat_col].notna()].copy()
    
    test_count = test_city_idx.sum()
    train_count = len(train_city)
    
    if train_count == 0:
        print(f"城市 {city}: Train中无数据，跳过（Test有 {test_count} 条）")
        continue
    
    if test_count == 0:
        continue
    
    # 提取经纬度
    test_coords = df_test.loc[test_city_idx, [lon_col, lat_col]].values
    train_coords = train_city[[lon_col, lat_col]].values
    
    # 计算距离矩阵
    distances = cdist(test_coords, train_coords, metric='euclidean')
    
    # 找到每个test样本最近的train样本索引
    nearest_train_idx = np.argmin(distances, axis=1)
    
    # 获取最近的train记录的聚类变量
    matched_features = train_city.iloc[nearest_train_idx][cluster_cols].values
    
    # 赋值给test数据
    df_test.loc[test_city_idx, cluster_cols] = matched_features
    
    total_matched += test_count
    
    print(f"城市 {city}: 匹配了 {test_count} 条Test数据（Train有 {train_count} 条）")

print(f"\n{'='*50}")
print(f"匹配完成！")
print(f"成功匹配: {total_matched} / {len(df_test)}")
print(f"处理后Test数据: {df_test.shape}")

# 验证聚类变量
print(f"\n{'='*50}")
print(f"聚类变量验证:")
non_zero_total = (df_test[cluster_cols] != 0).sum().sum()
has_cluster = (df_test[cluster_cols] != 0).any(axis=1).sum()
print(f"  非零值总数: {non_zero_total}")
print(f"  至少有1个非零cluster的样本数: {has_cluster}")

# 显示前几个样本的cluster分配
print(f"\n前5个样本的cluster分配:")
for idx in df_test.head().index:
    non_zero = df_test.loc[idx, cluster_cols][df_test.loc[idx, cluster_cols] != 0]
    if len(non_zero) > 0:
        print(f"  样本 {idx}: {non_zero.index[0]} = {non_zero.values[0]}")
    else:
        print(f"  样本 {idx}: 无cluster分配（可能经纬度缺失）")

# 保存
output = "ruc_Class25Q2_test_rent8.csv"
try:
    df_test.to_csv(output, index=False, encoding='utf-8-sig')
    print(f"\n✅ 已保存: {output}")
except PermissionError:
    output = "ruc_Class25Q2_test_rent8_new.csv"
    df_test.to_csv(output, index=False, encoding='utf-8-sig')
    print(f"\n✅ 原文件被占用，已保存为: {output}")


Test数据: (9773, 47)
Train数据: (97709, 305)

找到 256 个聚类变量
聚类变量范围: cluster_missing 到 cluster_299

城市列: 城市
Test城市分布:
城市
0     1512
1      567
2      936
3     1802
4     1168
5      592
6      188
7      837
8      668
9      340
10    1070
11      93
Name: count, dtype: int64

经纬度列: lon, lat

开始匹配，共 12 个城市...

城市 0: 匹配了 1512 条Test数据（Train有 14540 条）
城市 1: 匹配了 567 条Test数据（Train有 7725 条）
城市 2: 匹配了 936 条Test数据（Train有 12046 条）
城市 3: 匹配了 1802 条Test数据（Train有 15470 条）
城市 4: 匹配了 1168 条Test数据（Train有 11796 条）
城市 5: 匹配了 592 条Test数据（Train有 4075 条）
城市 6: 匹配了 188 条Test数据（Train有 1019 条）
城市 7: 匹配了 837 条Test数据（Train有 9441 条）
城市 8: 匹配了 668 条Test数据（Train有 7273 条）
城市 9: 匹配了 340 条Test数据（Train有 1255 条）
城市 10: 匹配了 1070 条Test数据（Train有 12658 条）
城市 11: 匹配了 93 条Test数据（Train有 411 条）

匹配完成！
成功匹配: 9773 / 9773
处理后Test数据: (9773, 303)

聚类变量验证:
  非零值总数: 9679
  至少有1个非零cluster的样本数: 9679

前5个样本的cluster分配:
  样本 0: cluster_51 = 1.0
  样本 1: cluster_212 = 1.0
  样本 2: cluster_60 = 1.0
  样本 3: cluster_268 = 1.0
  样本 4: cluster_121 =

In [9]:
# -*- coding: utf-8 -*-
"""
批量预测 - 四大月租模型（智能修正版 - 区分月租与总价）
"""

import pandas as pd
import numpy as np
import joblib
import warnings
import gc
import os
from datetime import datetime
warnings.filterwarnings('ignore')

print("=" * 80)
print("四模型批量预测 - 月租模型（智能修正版 - 区分月租与总价）")
print("=" * 80)

# ===================== 配置四个租金模型 =====================
MODELS = [
    {'name': 'OLS', 'model_file': 'rent_model_OLS_log_月租.pkl'},
    {'name': 'Ridge', 'model_file': 'rent_model_Ridge_log_月租.pkl'},
    {'name': 'Lasso', 'model_file': 'rent_model_Lasso_log_月租.pkl'},
    {'name': 'ElasticNet', 'model_file': 'rent_model_ElasticNet_log_月租.pkl'}
]

# ===================== 路径配置 =====================
path_test = r"H:\HW\ruc_Class25Q2_test_rent8.csv"
path_train = r"H:\HW\ruc_Class25Q2_train_rent91.csv"
output_dir = r"H:\HW"

scaler_path = 'rent_scaler_log_月租.pkl'
features_path = 'rent_selected_features_log_月租.pkl'

# ===================== 智能修正开关 =====================
enable_log_clipping = True       # ✅ 对数裁剪（防止极端值）
enable_quantile_calibration = True  # ✅ 分位数校准（对齐分布）
enable_outlier_report = True     # ✅ 异常值报告
correction_strength = 0.5        # 🎚️ 修正强度 (0-1，0.5=温和)


# ===================== 1. 加载公共资源 =====================
print("\nStep 1: 加载公共资源")
print("=" * 80)

try:
    scaler = joblib.load(scaler_path)
    selected_features = joblib.load(features_path)
    print(f"✅ Scaler + Features 加载完成")
    print(f"   Selected Features: {len(selected_features)} 个")
    
    if hasattr(scaler, 'feature_names_in_'):
        scaler_features = list(scaler.feature_names_in_)
        print(f"   Scaler特征: {len(scaler_features)} 个")
    else:
        scaler_features = []
        
except Exception as e:
    print(f"❌ 加载失败: {e}")
    exit()


# ===================== 2. 加载测试集 =====================
print("\nStep 2: 加载测试集")
print("=" * 80)

try:
    df_test_raw = pd.read_csv(path_test, encoding='utf-8')
    print(f"✅ 测试集: {df_test_raw.shape}")
    
    # ID检测
    id_col_name = None
    for col in ['ID', 'id', 'Id', 'index', '编号', 'Unnamed: 0']:
        if col in df_test_raw.columns:
            id_col_name = col
            break
    
    if id_col_name:
        test_ids = df_test_raw[id_col_name].copy()
        print(f"✅ ID列: '{id_col_name}'")
        df_test = df_test_raw.drop(columns=[id_col_name])
    else:
        test_ids = pd.Series(range(len(df_test_raw)), name='ID')
        print(f"⚠️ 自动生成ID")
        df_test = df_test_raw.copy()
    
    # 提取付款方式
    payment_cols = [col for col in df_test.columns if col.startswith('付款方式_')]
    
    if payment_cols:
        payment_info = df_test[payment_cols].copy()
        print(f"✅ 付款方式: {len(payment_cols)} 列")
        for col in payment_cols:
            count = payment_info[col].sum()
            print(f"   {col}: {count} ({count/len(payment_info)*100:.1f}%)")
    else:
        payment_info = None
        print(f"⚠️ 无付款方式信息")
    
except Exception as e:
    print(f"❌ 测试集加载失败: {e}")
    raise


# ===================== 3. 训练集统计（反推月租）=====================
print("\nStep 3: 训练集统计（反推月租）")
print("=" * 80)

try:
    # 读取训练集Price和付款方式
    cols_to_read = ['Price']
    df_train_peek = pd.read_csv(path_train, encoding='utf-8', nrows=0)
    train_payment_cols = [col for col in df_train_peek.columns if col.startswith('付款方式_')]
    cols_to_read.extend(train_payment_cols)
    
    df_train_stats = pd.read_csv(path_train, encoding='utf-8', usecols=cols_to_read)
    
    # 反推月租
    if train_payment_cols:
        payment_multiplier_train = np.ones(len(df_train_stats)) * 3  # 默认季付
        
        for col in train_payment_cols:
            if col in df_train_stats.columns:
                if '月付价' in col:
                    payment_multiplier_train[df_train_stats[col] == 1] = 1
                elif '双月付价' in col:
                    payment_multiplier_train[df_train_stats[col] == 1] = 2
                elif '季付价' in col:
                    payment_multiplier_train[df_train_stats[col] == 1] = 3
                elif '半年付价' in col:
                    payment_multiplier_train[df_train_stats[col] == 1] = 6
                elif '年付价' in col:
                    payment_multiplier_train[df_train_stats[col] == 1] = 12
        
        monthly_rent_train = df_train_stats['Price'] / payment_multiplier_train
    else:
        monthly_rent_train = df_train_stats['Price'] / 3
        print(f"⚠️ 训练集无付款方式，假设季付")
    
    # 原始空间统计
    train_mean = monthly_rent_train.mean()
    train_median = monthly_rent_train.median()
    train_std = monthly_rent_train.std()
    train_min = monthly_rent_train.min()
    train_max = monthly_rent_train.max()
    
    # 分位数统计
    train_quantiles = {
        'p01': monthly_rent_train.quantile(0.01),
        'p05': monthly_rent_train.quantile(0.05),
        'p10': monthly_rent_train.quantile(0.10),
        'p25': monthly_rent_train.quantile(0.25),
        'p50': monthly_rent_train.quantile(0.50),
        'p75': monthly_rent_train.quantile(0.75),
        'p90': monthly_rent_train.quantile(0.90),
        'p95': monthly_rent_train.quantile(0.95),
        'p99': monthly_rent_train.quantile(0.99)
    }
    
    print(f"📊 训练集【月租】统计（从Price反推）:")
    print(f"  样本数: {len(monthly_rent_train):,}")
    print(f"  范围: [{train_min:,.0f}, {train_max:,.0f}]")
    print(f"  均值: {train_mean:,.0f}  ±  {train_std:,.0f}")
    print(f"  中位数: {train_median:,.0f}")
    print(f"  分位数:")
    print(f"    P1:  {train_quantiles['p01']:>10,.0f}")
    print(f"    P5:  {train_quantiles['p05']:>10,.0f}")
    print(f"    P25: {train_quantiles['p25']:>10,.0f}")
    print(f"    P50: {train_quantiles['p50']:>10,.0f}")
    print(f"    P75: {train_quantiles['p75']:>10,.0f}")
    print(f"    P95: {train_quantiles['p95']:>10,.0f}")
    print(f"    P99: {train_quantiles['p99']:>10,.0f}")
    
    # 对数空间统计
    train_log = np.log1p(monthly_rent_train)
    train_log_mean = train_log.mean()
    train_log_std = train_log.std()
    train_log_min = train_log.min()
    train_log_max = train_log.max()
    
    # 对数空间的合理范围（±2σ）
    log_lower_bound = train_log_mean - 2 * train_log_std
    log_upper_bound = train_log_mean + 2 * train_log_std
    
    print(f"\n📊 训练集【log(月租+1)】统计:")
    print(f"  均值: {train_log_mean:.4f}  ±  {train_log_std:.4f}")
    print(f"  范围: [{train_log_min:.4f}, {train_log_max:.4f}]")
    print(f"  ±2σ范围: [{log_lower_bound:.4f}, {log_upper_bound:.4f}]")
    print(f"  对应月租: [{np.expm1(log_lower_bound):,.0f}, {np.expm1(log_upper_bound):,.0f}]")
    
    # 训练集Price统计（对比用）
    train_price_mean = df_train_stats['Price'].mean()
    train_price_median = df_train_stats['Price'].median()
    print(f"\n📊 训练集【Price总价】统计（对比参考）:")
    print(f"  均值: {train_price_mean:,.0f}")
    print(f"  中位数: {train_price_median:,.0f}")
    
    del df_train_stats
    gc.collect()
    
except Exception as e:
    print(f"❌ 训练集统计失败: {e}")
    import traceback
    traceback.print_exc()
    exit()


# ===================== 4. 特征工程 =====================
print("\nStep 4: 特征工程")
print("=" * 80)

def prepare_test_data(df):
    """特征工程"""
    df = df.copy()
    
    # 删除不需要的列
    cols_to_drop = [c for c in ['城市', '交易时间'] if c in df.columns]
    if cols_to_drop:
        df = df.drop(columns=cols_to_drop)
    
    # 删除城市特征
    city_cols = [col for col in df.columns if col.startswith('city_')]
    if city_cols:
        df = df.drop(columns=city_cols)
    
    # 处理'室'
    if '室' in df.columns:
        df['室'] = df['室'].fillna(df['室'].median())
    
    # 对数特征
    for feat in selected_features:
        if feat.endswith('_log'):
            original_feat = feat.replace('_log', '')
            if original_feat in df.columns:
                df[original_feat] = df[original_feat].clip(lower=0.01)
                df[feat] = np.log1p(df[original_feat])
    
    # 交互特征
    area_col = '面积' if '面积' in df.columns else '建筑面积'
    if area_col in df.columns and '室' in df.columns:
        df['室_adjusted'] = df['室'].replace(0, 1).fillna(2)
        if '建筑面积_per_室' in selected_features:
            df['建筑面积_per_室'] = df[area_col] / df['室_adjusted']
        if '面积_per_室' in selected_features:
            df['面积_per_室'] = df[area_col] / df['室_adjusted']
        df = df.drop(columns=['室_adjusted'], errors='ignore')
    
    # 面积分箱
    if area_col in df.columns:
        df['面积_小户型'] = (df[area_col] < 70).astype(int)
        df['面积_中户型'] = ((df[area_col] >= 70) & (df[area_col] <= 120)).astype(int)
        df['面积_大户型'] = (df[area_col] > 120).astype(int)
    
    # 特征对齐
    for f in selected_features:
        if f not in df.columns:
            df[f] = 0
    
    return df[selected_features].fillna(0)

X_test_base = prepare_test_data(df_test)
print(f"✅ 特征工程完成: {X_test_base.shape}")


# ===================== 5. 标准化 =====================
print("\nStep 5: 标准化")
print("=" * 80)

if len(scaler_features) > 0:
    scaler_features_clean = [f for f in scaler_features if not f.startswith('city_')]
    
    X_for_scaling = pd.DataFrame()
    for f in scaler_features_clean:
        if f in X_test_base.columns:
            X_for_scaling[f] = X_test_base[f]
        else:
            X_for_scaling[f] = 0
    
    X_for_scaling = X_for_scaling[scaler_features_clean].fillna(0)
    
    try:
        scaled_data = scaler.transform(X_for_scaling)
        
        X_test_scaled = X_test_base.copy()
        for i, f in enumerate(scaler_features_clean):
            if f in X_test_scaled.columns:
                X_test_scaled[f] = scaled_data[:, i]
        
        print(f"✅ 标准化完成")
        
    except Exception as e:
        print(f"❌ 标准化失败: {e}")
        X_test_scaled = X_test_base
else:
    X_test_scaled = X_test_base
    print(f"ℹ️ 无需标准化")

del X_test_base, df_test
gc.collect()


# ===================== 6. 智能修正函数 =====================
def smart_correction(y_pred_log_raw, train_stats, strength=0.5):
    """
    智能修正函数（基于月租对数空间）
    
    参数:
        y_pred_log_raw: 原始对数预测
        train_stats: 训练集统计信息
        strength: 修正强度 (0-1)
    
    返回:
        y_pred_log_corrected: 修正后的对数预测
        correction_report: 修正报告
    """
    y_pred_log = y_pred_log_raw.copy()
    n_total = len(y_pred_log)
    
    report = {
        'original_mean': y_pred_log.mean(),
        'original_median': np.median(y_pred_log),
        'original_std': y_pred_log.std(),
        'corrections': []
    }
    
    # ========== 1. 对数裁剪（防止极端值）==========
    if enable_log_clipping:
        log_mean = train_stats['log_mean']
        log_std = train_stats['log_std']
        
        # ±2.5σ范围（比训练时宽松）
        clip_lower = log_mean - 2.5 * log_std
        clip_upper = log_mean + 2.5 * log_std
        
        n_clipped_low = (y_pred_log < clip_lower).sum()
        n_clipped_high = (y_pred_log > clip_upper).sum()
        
        if n_clipped_low + n_clipped_high > 0:
            y_pred_log_clipped = np.clip(y_pred_log, clip_lower, clip_upper)
            
            # 温和过渡（strength控制）
            y_pred_log = strength * y_pred_log_clipped + (1 - strength) * y_pred_log
            
            report['corrections'].append({
                'step': '对数裁剪',
                'clipped_low': n_clipped_low,
                'clipped_high': n_clipped_high,
                'bounds': (clip_lower, clip_upper)
            })
    
    # ========== 2. 分位数校准（对齐分布）==========
    if enable_quantile_calibration:
        # 计算预测的分位数
        pred_quantiles = {
            'p25': np.percentile(y_pred_log, 25),
            'p50': np.percentile(y_pred_log, 50),
            'p75': np.percentile(y_pred_log, 75)
        }
        
        # 训练集的对应分位数（对数空间）
        train_log_quantiles = {
            'p25': np.log1p(train_stats['quantiles']['p25']),
            'p50': np.log1p(train_stats['quantiles']['p50']),
            'p75': np.log1p(train_stats['quantiles']['p75'])
        }
        
        # 计算偏移量
        shift_p50 = train_log_quantiles['p50'] - pred_quantiles['p50']
        
        # 温和校准（只调整中位数，保持分布形状）
        if abs(shift_p50) > 0.1:  # 只在偏差明显时校准
            y_pred_log_shifted = y_pred_log + strength * shift_p50
            y_pred_log = y_pred_log_shifted
            
            report['corrections'].append({
                'step': '中位数校准',
                'shift': shift_p50,
                'strength': strength
            })
    
    report['corrected_mean'] = y_pred_log.mean()
    report['corrected_median'] = np.median(y_pred_log)
    report['corrected_std'] = y_pred_log.std()
    
    return y_pred_log, report


# ===================== 7. 批量预测（含智能修正）=====================
print("\n" + "=" * 80)
print("开始批量预测（智能修正版 - 区分月租与总价）")
print("=" * 80)

# 准备训练集统计
train_stats_dict = {
    'mean': train_mean,
    'median': train_median,
    'std': train_std,
    'log_mean': train_log_mean,
    'log_std': train_log_std,
    'quantiles': train_quantiles
}

# 存储结果（区分月租和总价）
results_monthly_rent = {}  # 月租
results_price = {}         # 总价（月租*付款周期）
correction_reports = {}

for model_config in MODELS:
    model_name = model_config['name']
    model_file = model_config['model_file']
    
    print(f"\n{'='*60}")
    print(f"模型: {model_name}")
    print(f"{'='*60}")
    
    try:
        model = joblib.load(model_file)
        print(f"✅ 已加载")
        
        # 特征对齐
        if hasattr(model, 'feature_names_in_'):
            model_features = list(model.feature_names_in_)
            
            X_test_aligned = pd.DataFrame()
            for feat in model_features:
                if feat in X_test_scaled.columns:
                    X_test_aligned[feat] = X_test_scaled[feat]
                else:
                    X_test_aligned[feat] = 0
            
            X_test_final = X_test_aligned.fillna(0)
        else:
            X_test_final = X_test_scaled.fillna(0)
        
        # 🔥 原始预测（对数空间）
        y_pred_log_raw = model.predict(X_test_final)
        
        print(f"\n📊 原始对数预测:")
        print(f"  均值: {y_pred_log_raw.mean():.4f}")
        print(f"  中位数: {np.median(y_pred_log_raw):.4f}")
        print(f"  标准差: {y_pred_log_raw.std():.4f}")
        print(f"  范围: [{y_pred_log_raw.min():.4f}, {y_pred_log_raw.max():.4f}]")
        
        # 🔥 智能修正
        y_pred_log_corrected, correction_report = smart_correction(
            y_pred_log_raw, 
            train_stats_dict, 
            strength=correction_strength
        )
        
        print(f"\n✨ 修正后对数预测:")
        print(f"  均值: {y_pred_log_corrected.mean():.4f}")
        print(f"  中位数: {np.median(y_pred_log_corrected):.4f}")
        print(f"  标准差: {y_pred_log_corrected.std():.4f}")
        print(f"  范围: [{y_pred_log_corrected.min():.4f}, {y_pred_log_corrected.max():.4f}]")
        
        if correction_report['corrections']:
            print(f"\n🔧 应用的修正:")
            for corr in correction_report['corrections']:
                print(f"  - {corr['step']}")
                if 'shift' in corr:
                    print(f"    偏移: {corr['shift']:.4f} (强度={corr['strength']})")
                if 'clipped_low' in corr:
                    print(f"    裁剪: 低={corr['clipped_low']}, 高={corr['clipped_high']}")
        
        # 反变换到原始空间（月租）
        monthly_rent_corrected = np.expm1(y_pred_log_corrected)
        
        print(f"\n📊 【月租】预测统计:")
        print(f"  均值: {monthly_rent_corrected.mean():,.0f}")
        print(f"  中位数: {np.median(monthly_rent_corrected):,.0f}")
        print(f"  标准差: {monthly_rent_corrected.std():,.0f}")
        print(f"  范围: [{monthly_rent_corrected.min():,.0f}, {monthly_rent_corrected.max():,.0f}]")
        
        # 还原总价（Price = 月租 * 付款周期）
        if payment_info is not None:
            payment_multiplier = np.ones(len(monthly_rent_corrected)) * 3  # 默认季付
            
            for col in payment_cols:
                if '月付价' in col:
                    payment_multiplier[payment_info[col] == 1] = 1
                elif '双月付价' in col:
                    payment_multiplier[payment_info[col] == 1] = 2
                elif '季付价' in col:
                    payment_multiplier[payment_info[col] == 1] = 3
                elif '半年付价' in col:
                    payment_multiplier[payment_info[col] == 1] = 6
                elif '年付价' in col:
                    payment_multiplier[payment_info[col] == 1] = 12
            
            price_corrected = monthly_rent_corrected * payment_multiplier
        else:
            payment_multiplier = 3
            price_corrected = monthly_rent_corrected * 3
        
        print(f"\n📊 【Price总价】预测统计:")
        print(f"  均值: {price_corrected.mean():,.0f}")
        print(f"  中位数: {np.median(price_corrected):,.0f}")
        print(f"  范围: [{price_corrected.min():,.0f}, {price_corrected.max():,.0f}]")
        
        # vs训练集（基于月租对比）
        mean_diff_corrected = (monthly_rent_corrected.mean() - train_mean) / train_mean * 100
        median_diff_corrected = (np.median(monthly_rent_corrected) - train_median) / train_median * 100
        print(f"\n📈 vs训练集（月租维度）:")
        print(f"  均值差异: {mean_diff_corrected:+.2f}%")
        print(f"  中位数差异: {median_diff_corrected:+.2f}%")
        
        # 异常值报告（基于月租）
        if enable_outlier_report:
            too_low = (monthly_rent_corrected < train_quantiles['p01']).sum()
            too_high = (monthly_rent_corrected > train_quantiles['p99']).sum()
            print(f"\n⚠️ 异常值检测（月租，相对训练集P1-P99范围）:")
            print(f"  < P1 ({train_quantiles['p01']:,.0f}): {too_low} ({too_low/len(monthly_rent_corrected)*100:.1f}%)")
            print(f"  > P99 ({train_quantiles['p99']:,.0f}): {too_high} ({too_high/len(monthly_rent_corrected)*100:.1f}%)")
        
        # 保存结果
        results_monthly_rent[model_name] = monthly_rent_corrected
        results_price[model_name] = price_corrected
        correction_reports[model_name] = correction_report
        
        print(f"✅ {model_name} 完成")
        
    except Exception as e:
        print(f"❌ {model_name} 预测失败: {e}")
        import traceback
        traceback.print_exc()
        continue
    
    gc.collect()


# ===================== 8. 保存文件（区分月租和总价）=====================
print("\n" + "=" * 80)
print("保存预测文件")
print("=" * 80)

for model_name in results_monthly_rent.keys():
    monthly_rent_pred = results_monthly_rent[model_name]
    price_pred = results_price[model_name]
    
    # 1. 保存完整版（ID + 月租 + 总价）
    output_full = os.path.join(output_dir, f"rent_pred_{model_name}_smart_full.csv")
    df_full = pd.DataFrame({
        test_ids.name: test_ids,
        'MonthlyRent': monthly_rent_pred,
        'Price': price_pred
    })
    
    try:
        df_full.to_csv(output_full, index=False, encoding='utf-8-sig')
        print(f"✅ {model_name}: rent_pred_{model_name}_smart_full.csv (ID + 月租 + 总价)")
    except Exception as e:
        print(f"❌ {model_name} 保存失败: {e}")
    
    # 2. 保存提交版（仅Price，用于提交）
    output_submit = os.path.join(output_dir, f"rent_pred_{model_name}_smart_submit.csv")
    df_submit = pd.DataFrame({'Price': price_pred})
    
    try:
        df_submit.to_csv(output_submit, index=False, encoding='utf-8-sig')
        print(f"✅ {model_name}: rent_pred_{model_name}_smart_submit.csv (仅Price，提交用)")
    except Exception as e:
        print(f"❌ {model_name} 保存失败: {e}")


# ===================== 9. 对比分析（基于月租）=====================
print("\n" + "=" * 80)
print("四模型对比分析（基于【月租】维度）")
print("=" * 80)

if len(results_monthly_rent) > 0:
    comparison = []
    for model_name, monthly_rent in results_monthly_rent.items():
        comparison.append({
            '模型': model_name,
            '月租均值': f"{monthly_rent.mean():,.0f}",
            '月租中位数': f"{np.median(monthly_rent):,.0f}",
            '月租标准差': f"{monthly_rent.std():,.0f}",
            '月租最小值': f"{monthly_rent.min():,.0f}",
            '月租最大值': f"{monthly_rent.max():,.0f}",
            'vs训练集均值': f"{(monthly_rent.mean()/train_mean - 1)*100:+.2f}%",
            'vs训练集中位数': f"{(np.median(monthly_rent)/train_median - 1)*100:+.2f}%"
        })
    
    df_comparison = pd.DataFrame(comparison)
    print(df_comparison.to_string(index=False))
    
    # 保存对比表
    comparison_path = os.path.join(output_dir, "rent_pred_comparison_monthly_rent.csv")
    df_comparison.to_csv(comparison_path, index=False, encoding='utf-8-sig')
    print(f"\n✅ 已保存对比表: rent_pred_comparison_monthly_rent.csv")
    
    if len(results_monthly_rent) >= 2:
        pred_array = np.array([results_monthly_rent[name] for name in results_monthly_rent.keys()])
        pred_std = pred_array.std(axis=0).mean()
        pred_range = pred_array.max(axis=0) - pred_array.min(axis=0)
        
        print(f"\n📊 模型间差异（月租维度）:")
        print(f"  平均标准差: {pred_std:,.0f}")
        print(f"  平均极差: {pred_range.mean():,.0f}")
        print(f"  极差中位数: {np.median(pred_range):,.0f}")


# ===================== 10. Price总价对比（参考）=====================
print("\n" + "=" * 80)
print("四模型对比分析（基于【Price总价】维度 - 参考）")
print("=" * 80)

if len(results_price) > 0:
    comparison_price = []
    for model_name, price in results_price.items():
        comparison_price.append({
            '模型': model_name,
            'Price均值': f"{price.mean():,.0f}",
            'Price中位数': f"{np.median(price):,.0f}",
            'Price最小值': f"{price.min():,.0f}",
            'Price最大值': f"{price.max():,.0f}"
        })
    
    df_comparison_price = pd.DataFrame(comparison_price)
    print(df_comparison_price.to_string(index=False))


# ===================== 11. 修正报告 =====================
print("\n" + "=" * 80)
print("智能修正报告（对数空间）")
print("=" * 80)

for model_name, report in correction_reports.items():
    print(f"\n{model_name}:")
    print(f"  原始 - 均值: {report['original_mean']:.4f}, 中位数: {report['original_median']:.4f}")
    print(f"  修正 - 均值: {report['corrected_mean']:.4f}, 中位数: {report['corrected_median']:.4f}")
    print(f"  变化 - 均值: {report['corrected_mean'] - report['original_mean']:+.4f}")
    print(f"  变化 - 中位数: {report['corrected_median'] - report['original_median']:+.4f}")


# ===================== 12. 总结 =====================
print("\n" + "=" * 80)
print("🎉 智能预测完成！")
print("=" * 80)
print(f"\n💡 关键说明:")
print(f"  1. 【月租】= 模型直接预测目标（所有统计对比基于此）")
print(f"  2. 【Price】= 月租 × 付款周期（用于提交）")
print(f"  3. 训练集月租统计（参考基准）:")
print(f"     - 均值: {train_mean:,.0f}")
print(f"     - 中位数: {train_median:,.0f}")
print(f"     - P1-P99: [{train_quantiles['p01']:,.0f}, {train_quantiles['p99']:,.0f}]")
print(f"\n🔧 智能修正配置:")
print(f"  - 对数裁剪: {'✅' if enable_log_clipping else '❌'} (±2.5σ)")
print(f"  - 分位数校准: {'✅' if enable_quantile_calibration else '❌'}")
print(f"  - 修正强度: {correction_strength} (0=不修正, 1=完全修正)")
print(f"\n📁 输出文件:")
print(f"  - *_smart_full.csv: 完整版（ID + 月租 + 总价）")
print(f"  - *_smart_submit.csv: 提交版（仅Price）")
print(f"  - rent_pred_comparison_monthly_rent.csv: 模型对比表（月租维度）")
print("=" * 80)

四模型批量预测 - 月租模型（智能修正版 - 区分月租与总价）

Step 1: 加载公共资源
✅ Scaler + Features 加载完成
   Selected Features: 303 个
   Scaler特征: 6 个

Step 2: 加载测试集
✅ 测试集: (9773, 303)
⚠️ 自动生成ID
✅ 付款方式: 4 列
   付款方式_双月付价: 2 (0.0%)
   付款方式_季付价: 6537 (66.9%)
   付款方式_年付价: 222 (2.3%)
   付款方式_月付价: 2317 (23.7%)

Step 3: 训练集统计（反推月租）
📊 训练集【月租】统计（从Price反推）:
  样本数: 97,709
  范围: [2,903, 15,230,042]
  均值: 296,279  ±  389,137
  中位数: 183,975
  分位数:
    P1:      15,471
    P5:      36,059
    P25:     80,293
    P50:    183,975
    P75:    354,344
    P95:    948,503
    P99:  1,775,890

📊 训练集【log(月租+1)】统计:
  均值: 12.0745  ±  1.0303
  范围: [7.9739, 16.5388]
  ±2σ范围: [10.0139, 14.1351]
  对应月租: [22,333, 1,376,508]

📊 训练集【Price总价】统计（对比参考）:
  均值: 586,986
  中位数: 400,489

Step 4: 特征工程
✅ 特征工程完成: (9773, 303)

Step 5: 标准化
✅ 标准化完成

开始批量预测（智能修正版 - 区分月租与总价）

模型: OLS
✅ 已加载

📊 原始对数预测:
  均值: 12.3060
  中位数: 12.2604
  标准差: 0.7837
  范围: [8.6462, 13.8668]

✨ 修正后对数预测:
  均值: 12.2423
  中位数: 12.1915
  标准差: 0.7616
  范围: [9.0035, 13.7979]

🔧 应用的修正:
  - 对数裁剪
  